# Cleaning and organization of data


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_MARKERS = {"01_raw_data", "02_clean_data", "03_outputs", "notebooks"}

BASE_DIR = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if PROJECT_MARKERS.issubset({p.name for p in path.iterdir() if p.is_dir()})
)

RAW_DIR = BASE_DIR / "01_raw_data"
CLEAN_DIR = BASE_DIR / "02_clean_data"

IN_RANGE_DIR = CLEAN_DIR / "2010_2025"
IMPLAUSIBLE_DIR = CLEAN_DIR / "implausible_2010_2025"
OUT_OF_RANGE_DIR = CLEAN_DIR / "outside_2010_2025"

CLEAN_DIR.mkdir(exist_ok=True)
IN_RANGE_DIR.mkdir(exist_ok=True)
IMPLAUSIBLE_DIR.mkdir(exist_ok=True)
OUT_OF_RANGE_DIR.mkdir(exist_ok=True)


START_DATE = pd.Timestamp("2010-01-01")
END_DATE = pd.Timestamp("2025-12-31 23:59:59")


rename_map = {
    "Patient DOB Year": "patient_dob_year",
    "Acc de-ident": "accession_id",
    "Acc de-ident ": "accession_id",
    "Acc de-indent": "accession_id",
    "ID de-ident": "patient_id",
    "ID de-indent": "patient_id",
    "Collected": "collected_raw",
    "ResultName": "result_name",
    "ResultCode": "result_code",
    "Result": "result_raw",
    "Units": "units",
}


def clean_result_column(result_series):
    s = result_series.astype("string").str.strip()

    extracted = s.str.extract(
        r"^\s*(?P<modifier>[<>])?\s*(?P<number>\d+(?:\.\d+)?)\s*(?P<extra>.*)?$"
    )

    out = pd.DataFrame(index=result_series.index)

    out["result_raw"] = s
    out["result_modifier"] = extracted["modifier"]
    out["result_is_below_limit"] = out["result_modifier"].eq("<").fillna(False)
    out["result_is_above_limit"] = out["result_modifier"].eq(">").fillna(False)

    out["result_numeric"] = pd.to_numeric(
        extracted["number"],
        errors="coerce",
    )

    extra = extracted["extra"].astype("string").str.strip()
    out["result_extra_text"] = extra.mask(extra.eq(""), pd.NA)

    out["result_detection_limit"] = np.where(
        out["result_is_below_limit"] | out["result_is_above_limit"],
        out["result_numeric"],
        np.nan,
    )

    out["result_for_analysis"] = out["result_numeric"]
    out.loc[out["result_is_below_limit"], "result_for_analysis"] = (
        out.loc[out["result_is_below_limit"], "result_detection_limit"] / 2
    )

    return out


def add_implausible_flags(df):
    flags = pd.Series("", index=df.index, dtype="string")

    def add_flag(mask, reason):
        nonlocal flags
        mask = mask.fillna(False)

        flags.loc[mask] = flags.loc[mask].where(
            flags.loc[mask].eq(""),
            flags.loc[mask] + "|",
        ) + reason

    if "patient_id" in df.columns:
        add_flag(
            df["patient_id"].isna() | df["patient_id"].eq(""),
            "missing_patient_id",
        )

    if "accession_id" in df.columns:
        add_flag(
            df["accession_id"].isna() | df["accession_id"].eq(""),
            "missing_accession_id",
        )

    if "patient_dob_year" in df.columns:
        add_flag(df["patient_dob_year"].isna(), "invalid_dob_year")
        add_flag(df["patient_dob_year"] < 1900, "dob_year_before_1900")

    if {"patient_dob_year", "collected_datetime"}.issubset(df.columns):
        add_flag(
            df["patient_dob_year"] > df["collected_datetime"].dt.year,
            "dob_year_after_collection",
        )

    if "collected_datetime" in df.columns:
        add_flag(
            df["collected_datetime"].isna(),
            "invalid_collected_datetime",
        )

    if "age_at_collection" in df.columns:
        add_flag(df["age_at_collection"] < 0, "negative_age")
        add_flag(df["age_at_collection"] > 120, "age_over_120")

    if "result_numeric" in df.columns:
        add_flag(df["result_numeric"].isna(), "unparseable_result")
        add_flag(df["result_numeric"] < 0, "negative_result")

    if "result_extra_text" in df.columns:
        add_flag(df["result_extra_text"].notna(), "result_has_extra_text")

    if "units" in df.columns:
        add_flag(
            df["units"].isna() | df["units"].eq(""),
            "missing_units",
        )

    return flags.replace("", pd.NA)


def clean_one_file(file_path):
    df = pd.read_excel(file_path, sheet_name=0, dtype=str)
    df = df.rename(columns=rename_map).copy()

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype("string").str.strip()

    if "patient_dob_year" in df.columns:
        df["patient_dob_year"] = pd.to_numeric(
            df["patient_dob_year"],
            errors="coerce",
        ).astype("Int64")

    if "collected_raw" in df.columns:
        df["collected_datetime"] = pd.to_datetime(
            df["collected_raw"],
            errors="coerce",
        )
        df["collected_date"] = df["collected_datetime"].dt.strftime("%Y-%m-%d")
        df["collected_time"] = df["collected_datetime"].dt.strftime("%H:%M:%S")

    if {"patient_dob_year", "collected_datetime"}.issubset(df.columns):
        df["age_at_collection"] = (
            df["collected_datetime"].dt.year - df["patient_dob_year"]
        ).astype("Int64")

    if "result_raw" in df.columns:
        result_clean = clean_result_column(df["result_raw"])
        df = df.drop(columns=["result_raw"])
        df = pd.concat([df, result_clean], axis=1)

    df["source_file"] = file_path.name
    df["implausible_flags"] = add_implausible_flags(df)
    df["is_implausible"] = df["implausible_flags"].notna()

    return df


for file_path in sorted(RAW_DIR.glob("*.xlsx")):
    df_clean = clean_one_file(file_path)

    base_name = (
        file_path.stem
        .replace("RAW_EXPORT_PREFIX_", "")
        .replace(" ", "_")
    )

    in_range = df_clean[
        (df_clean["collected_datetime"] >= START_DATE)
        & (df_clean["collected_datetime"] <= END_DATE)
    ].copy()

    outside_range = df_clean[
        (df_clean["collected_datetime"] < START_DATE)
        | (df_clean["collected_datetime"] > END_DATE)
        | (df_clean["collected_datetime"].isna())
    ].copy()

    plausible_in_range = in_range[
        in_range["is_implausible"] == False
    ].copy()

    implausible_in_range = in_range[
        in_range["is_implausible"] == True
    ].copy()

    all_clean_file = CLEAN_DIR / f"{base_name}_all_clean.csv"
    in_range_file = IN_RANGE_DIR / f"{base_name}_2010_2025_clean.csv"
    implausible_file = IMPLAUSIBLE_DIR / f"{base_name}_2010_2025_implausible.csv"
    outside_range_file = OUT_OF_RANGE_DIR / f"{base_name}_outside_2010_2025.csv"

    df_clean.to_csv(all_clean_file, index=False)
    plausible_in_range.to_csv(in_range_file, index=False)
    implausible_in_range.to_csv(implausible_file, index=False)
    outside_range.to_csv(outside_range_file, index=False)

    print(f"File: {file_path.name}")
    print(f"  All cleaned rows: {len(df_clean):,}")
    print(f"  2010-2025 plausible rows: {len(plausible_in_range):,}")
    print(f"  2010-2025 implausible rows: {len(implausible_in_range):,}")
    print(f"  Outside 2010-2025 rows: {len(outside_range):,}")
    print()

# Separation of non-repeat and repeat patients


In [ ]:
from pathlib import Path

import pandas as pd


BASE_DIR = Path.cwd().resolve()

while not {"01_raw_data", "02_clean_data", "03_outputs_v2", "notebooks"}.issubset(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()}
):
    BASE_DIR = BASE_DIR.parent


CLEAN_DIR = BASE_DIR / "02_clean_data"
OUTPUT_DIR = BASE_DIR / "03_outputs_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


date_col = "collected_date"
patient_id_col = "patient_id"
result_col = "result_for_analysis"

start_year = 2010
end_year = 2025


def classify_test_status(value):
    if value >= 0.35:
        return "Positive"
    if value < 0.10:
        return "Negative"
    return "Borderline"


def classify_transition(statuses):
    statuses = list(statuses)
    unique_statuses = set(statuses)

    if len(unique_statuses) == 1:
        return f"Stayed {statuses[0].lower()}"

    first_status = statuses[0]
    last_status = statuses[-1]

    return f"{first_status} to {last_status}"


for input_file in sorted(CLEAN_DIR.glob("*_clean.csv")):
    base_name = input_file.stem.replace("_clean", "")

    non_repeated_output_file = (
        OUTPUT_DIR / f"{base_name}_non_repeated_patient_tests_2010_2025.csv"
    )

    repeated_output_file = (
        OUTPUT_DIR / f"{base_name}_repeated_patients_one_row_per_patient_2010_2025.csv"
    )

    df = pd.read_csv(input_file, low_memory=False)

    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df["year"] = df[date_col].dt.year
    df[result_col] = pd.to_numeric(df[result_col], errors="coerce")

    df = df.dropna(subset=[patient_id_col, date_col, result_col, "year"]).copy()
    df["year"] = df["year"].astype(int)
    df = df[df["year"].between(start_year, end_year)].copy()

    df["test_status"] = df[result_col].apply(classify_test_status)

    tests_per_patient = df.groupby(patient_id_col).size()

    non_repeated_patient_ids = tests_per_patient[tests_per_patient == 1].index
    repeated_patient_ids = tests_per_patient[tests_per_patient > 1].index

    non_repeated_df = df[
        df[patient_id_col].isin(non_repeated_patient_ids)
    ].copy()

    repeated_df = df[
        df[patient_id_col].isin(repeated_patient_ids)
    ].copy()

    repeated_df = repeated_df.sort_values([patient_id_col, date_col]).copy()

    rows = []

    for patient_id, sub in repeated_df.groupby(patient_id_col, sort=True):
        sub = sub.sort_values(date_col).reset_index(drop=True)

        row = {
            "patient_id": patient_id,
            "n_tests_2010_2025": len(sub),
            "first_test_date": sub[date_col].iloc[0].date().isoformat(),
            "last_test_date": sub[date_col].iloc[-1].date().isoformat(),
            "first_status": sub["test_status"].iloc[0],
            "last_status": sub["test_status"].iloc[-1],
            "transition_summary": classify_transition(sub["test_status"]),
            "status_sequence": " -> ".join(sub["test_status"].tolist()),
            "result_sequence": " | ".join(
                [f"{x:.2f}" for x in sub[result_col].tolist()]
            ),
        }

        for i, (_, r) in enumerate(sub.iterrows(), start=1):
            row[f"test_{i}_date"] = r[date_col].date().isoformat()
            row[f"test_{i}_year"] = int(r["year"])
            row[f"test_{i}_result"] = round(float(r[result_col]), 2)
            row[f"test_{i}_status"] = r["test_status"]

        rows.append(row)

    repeated_patients_table = pd.DataFrame(rows)

    non_repeated_df.to_csv(non_repeated_output_file, index=False)
    repeated_patients_table.to_csv(repeated_output_file, index=False)

    print("=" * 80)
    print(f"Input file: {input_file.name}")
    print(f"Saved non-repeated patients to: {non_repeated_output_file.name}")
    print(f"Saved repeated patients to: {repeated_output_file.name}")
    print(f"Total 2010-2025 tests: {len(df):,}")
    print(f"Unique patients 2010-2025: {df[patient_id_col].nunique():,}")
    print(f"Non-repeated patients: {len(non_repeated_patient_ids):,}")
    print(f"Repeated patients: {len(repeated_patient_ids):,}")

### Categorizing repeated patients


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd().resolve()

while not {"01_raw_data", "02_clean_data", "03_outputs_v2", "notebooks"}.issubset(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()}
):
    BASE_DIR = BASE_DIR.parent

CLEAN_DIR = BASE_DIR / "02_clean_data"
OUTPUT_DIR = BASE_DIR / "03_outputs_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_file = CLEAN_DIR / "aGal_IgE_20260406_clean.csv"

output_file = (
    OUTPUT_DIR
    / "aGal_IgE_repeated_patients_audit_with_same_day_and_longitudinal_flags_2010_2025.csv"
)

DATE_COL = "collected_date"
PATIENT_ID_COL = "patient_id"
DOB_COL = "patient_dob_year"
IGE_COL = "result_for_analysis"

MIN_YEAR = 2010
MAX_YEAR = 2025
POSITIVE_CUTOFF = 0.35

def fmt_date(x):
    if pd.isna(x):
        return ""
    return pd.to_datetime(x).date().isoformat()


def status_from_ige(x):
    return "Positive" if x >= POSITIVE_CUTOFF else "Non-positive"


def get_mode_or_nan(x):
    x = x.dropna()
    if len(x) == 0:
        return np.nan
    return x.mode().iloc[0]


def classify_transition(statuses):
    statuses = list(statuses)
    unique_statuses = set(statuses)

    if unique_statuses == {"Positive"}:
        return "Stayed positive"

    if unique_statuses == {"Non-positive"}:
        return "Stayed non-positive"

    first_status = statuses[0]
    last_status = statuses[-1]

    if first_status == "Non-positive" and last_status == "Positive":
        if statuses == sorted(statuses, key=lambda x: 0 if x == "Non-positive" else 1):
            return "Non-positive to positive"
        return "Fluctuated, ended positive"

    if first_status == "Positive" and last_status == "Non-positive":
        if statuses == sorted(statuses, key=lambda x: 0 if x == "Positive" else 1):
            return "Positive to non-positive"
        return "Fluctuated, ended non-positive"

    return "Fluctuated"


df = pd.read_csv(input_file, low_memory=False)

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df["collected_day"] = df[DATE_COL].dt.normalize()
df["collection_year"] = df[DATE_COL].dt.year

df[PATIENT_ID_COL] = df[PATIENT_ID_COL].astype("string").str.strip()
df[DOB_COL] = pd.to_numeric(df[DOB_COL], errors="coerce")
df[IGE_COL] = pd.to_numeric(df[IGE_COL], errors="coerce")

df = df.dropna(
    subset=[PATIENT_ID_COL, DATE_COL, "collected_day", "collection_year", IGE_COL]
).copy()

df = df[df[PATIENT_ID_COL] != ""].copy()
df["collection_year"] = df["collection_year"].astype(int)
df = df[df["collection_year"].between(MIN_YEAR, MAX_YEAR)].copy()

df["raw_test_status"] = np.where(
    df[IGE_COL] >= POSITIVE_CUTOFF,
    "Positive",
    "Non-positive",
)

raw_tests_per_patient = df.groupby(PATIENT_ID_COL).size()
repeated_patient_ids = raw_tests_per_patient[raw_tests_per_patient > 1].index

rep_raw = df[df[PATIENT_ID_COL].isin(repeated_patient_ids)].copy()

day_level = (
    rep_raw.groupby([PATIENT_ID_COL, "collected_day"], as_index=False)
    .agg(
        patient_dob_year=(DOB_COL, get_mode_or_nan),
        n_raw_records_that_day=(IGE_COL, "size"),
        alpha_gal_ige_patient_day_median=(IGE_COL, "median"),
        alpha_gal_ige_patient_day_min=(IGE_COL, "min"),
        alpha_gal_ige_patient_day_max=(IGE_COL, "max"),
        collection_year=("collection_year", "first"),
    )
)

day_level["patient_day_status"] = np.where(
    day_level["alpha_gal_ige_patient_day_median"] >= POSITIVE_CUTOFF,
    "Positive",
    "Non-positive",
)

day_level["had_multiple_records_that_day"] = day_level["n_raw_records_that_day"] > 1
day_level = day_level.sort_values([PATIENT_ID_COL, "collected_day"]).copy()

same_day_details = (
    day_level[day_level["had_multiple_records_that_day"]]
    .assign(
        same_day_detail=lambda x: (
            x["collected_day"].dt.date.astype(str)
            + " ("
            + x["n_raw_records_that_day"].astype(str)
            + " records; median IgE "
            + x["alpha_gal_ige_patient_day_median"].round(2).astype(str)
            + "; min "
            + x["alpha_gal_ige_patient_day_min"].round(2).astype(str)
            + "; max "
            + x["alpha_gal_ige_patient_day_max"].round(2).astype(str)
            + ")"
        )
    )
    .groupby(PATIENT_ID_COL)["same_day_detail"]
    .apply(lambda x: " | ".join(x))
    .rename("multiple_tests_same_day_details")
    .reset_index()
)

rows = []

for patient_id, raw_sub in rep_raw.groupby(PATIENT_ID_COL, sort=True):
    raw_date_only = raw_sub.sort_values(DATE_COL, kind="mergesort").reset_index(drop=True)
    raw_date_ige = raw_sub.sort_values([DATE_COL, IGE_COL], kind="mergesort").reset_index(drop=True)

    day_sub = day_level[day_level[PATIENT_ID_COL] == patient_id].copy()
    day_sub = day_sub.sort_values("collected_day").reset_index(drop=True)

    n_raw_records = len(raw_sub)
    n_distinct_dates = len(day_sub)

    raw_date_only_statuses = raw_date_only["raw_test_status"].tolist()
    raw_date_ige_statuses = raw_date_ige["raw_test_status"].tolist()
    patient_day_statuses = day_sub["patient_day_status"].tolist()

    has_multiple_tests_same_day = bool(day_sub["had_multiple_records_that_day"].any())
    eligible_for_longitudinal_analysis = n_distinct_dates >= 2
    same_day_only_repeat_patient = n_raw_records > 1 and n_distinct_dates == 1

    row = {
        "patient_id": patient_id,
        "patient_dob_year": get_mode_or_nan(raw_sub[DOB_COL]),

        "n_raw_tests_2010_2025": n_raw_records,
        "n_distinct_test_dates_2010_2025": n_distinct_dates,
        "eligible_for_longitudinal_analysis": eligible_for_longitudinal_analysis,
        "same_day_only_repeat_patient": same_day_only_repeat_patient,
        "has_multiple_tests_same_day": has_multiple_tests_same_day,
        "same_day_duplicate_plus_later_date": has_multiple_tests_same_day and eligible_for_longitudinal_analysis,
        "n_days_with_multiple_tests": int(day_sub["had_multiple_records_that_day"].sum()),
        "max_raw_records_on_same_day": int(day_sub["n_raw_records_that_day"].max()),

        "first_raw_date_only_date": fmt_date(raw_date_only[DATE_COL].iloc[0]),
        "first_raw_date_only_ige": round(float(raw_date_only[IGE_COL].iloc[0]), 2),
        "first_raw_date_only_status": raw_date_only["raw_test_status"].iloc[0],
        "last_raw_date_only_date": fmt_date(raw_date_only[DATE_COL].iloc[-1]),
        "last_raw_date_only_ige": round(float(raw_date_only[IGE_COL].iloc[-1]), 2),
        "last_raw_date_only_status": raw_date_only["raw_test_status"].iloc[-1],
        "raw_date_only_transition_summary": classify_transition(raw_date_only_statuses),

        "first_raw_date_ige_ascending_date": fmt_date(raw_date_ige[DATE_COL].iloc[0]),
        "first_raw_date_ige_ascending_ige": round(float(raw_date_ige[IGE_COL].iloc[0]), 2),
        "first_raw_date_ige_ascending_status": raw_date_ige["raw_test_status"].iloc[0],
        "raw_date_ige_ascending_transition_summary": classify_transition(raw_date_ige_statuses),

        "first_patient_day_date": fmt_date(day_sub["collected_day"].iloc[0]),
        "first_patient_day_ige_median": round(float(day_sub["alpha_gal_ige_patient_day_median"].iloc[0]), 2),
        "first_patient_day_status": day_sub["patient_day_status"].iloc[0],
        "last_patient_day_date": fmt_date(day_sub["collected_day"].iloc[-1]),
        "last_patient_day_ige_median": round(float(day_sub["alpha_gal_ige_patient_day_median"].iloc[-1]), 2),
        "last_patient_day_status": day_sub["patient_day_status"].iloc[-1],
        "patient_day_transition_summary": classify_transition(patient_day_statuses),

        "raw_date_only_status_sequence": " -> ".join(raw_date_only_statuses),
        "raw_date_only_ige_sequence": " | ".join([f"{x:.2f}" for x in raw_date_only[IGE_COL].tolist()]),
        "patient_day_status_sequence": " -> ".join(patient_day_statuses),
        "patient_day_median_ige_sequence": " | ".join(
            [f"{x:.2f}" for x in day_sub["alpha_gal_ige_patient_day_median"].tolist()]
        ),
    }

    for i, (_, r) in enumerate(day_sub.iterrows(), start=1):
        row[f"patient_day_{i}_date"] = fmt_date(r["collected_day"])
        row[f"patient_day_{i}_year"] = int(r["collection_year"])
        row[f"patient_day_{i}_ige_median"] = round(float(r["alpha_gal_ige_patient_day_median"]), 2)
        row[f"patient_day_{i}_ige_min"] = round(float(r["alpha_gal_ige_patient_day_min"]), 2)
        row[f"patient_day_{i}_ige_max"] = round(float(r["alpha_gal_ige_patient_day_max"]), 2)
        row[f"patient_day_{i}_n_raw_records"] = int(r["n_raw_records_that_day"])
        row[f"patient_day_{i}_status"] = r["patient_day_status"]

    rows.append(row)

out = pd.DataFrame(rows)

out = out.merge(
    same_day_details,
    on=PATIENT_ID_COL,
    how="left",
)

out["multiple_tests_same_day_details"] = out["multiple_tests_same_day_details"].fillna("")

front_cols = [
    "patient_id",
    "patient_dob_year",
    "n_raw_tests_2010_2025",
    "n_distinct_test_dates_2010_2025",
    "eligible_for_longitudinal_analysis",
    "same_day_only_repeat_patient",
    "has_multiple_tests_same_day",
    "same_day_duplicate_plus_later_date",
    "n_days_with_multiple_tests",
    "max_raw_records_on_same_day",
    "multiple_tests_same_day_details",
    "first_raw_date_only_date",
    "first_raw_date_only_ige",
    "first_raw_date_only_status",
    "first_raw_date_ige_ascending_date",
    "first_raw_date_ige_ascending_ige",
    "first_raw_date_ige_ascending_status",
    "first_patient_day_date",
    "first_patient_day_ige_median",
    "first_patient_day_status",
    "last_patient_day_date",
    "last_patient_day_ige_median",
    "last_patient_day_status",
    "patient_day_transition_summary",
]

other_cols = [c for c in out.columns if c not in front_cols]
out = out[front_cols + other_cols]

out.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")

print("\nKey counts you should be able to reproduce with filters:")
print(f"All rows, patients with >1 raw test record: {len(out):,}")
print(f"eligible_for_longitudinal_analysis == True: {int(out['eligible_for_longitudinal_analysis'].sum()):,}")
print(f"has_multiple_tests_same_day == True: {int(out['has_multiple_tests_same_day'].sum()):,}")
print(f"same_day_only_repeat_patient == True: {int(out['same_day_only_repeat_patient'].sum()):,}")
print(f"same_day_duplicate_plus_later_date == True: {int(out['same_day_duplicate_plus_later_date'].sum()):,}")

print(
    "first_raw_date_only_status == Positive: "
    f"{int((out['first_raw_date_only_status'] == 'Positive').sum()):,}"
)

print(
    "first_raw_date_ige_ascending_status == Positive: "
    f"{int((out['first_raw_date_ige_ascending_status'] == 'Positive').sum()):,}"
)

print(
    "eligible_for_longitudinal_analysis == True AND first_patient_day_status == Positive: "
    f"{int((out['eligible_for_longitudinal_analysis'] & (out['first_patient_day_status'] == 'Positive')).sum()):,}"
)

print("\nPatient-day transition summary:")
print(out["patient_day_transition_summary"].value_counts().to_string())

out.head()

# Basic overview of entire data set


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().resolve()

while not {"01_raw_data", "02_clean_data", "03_outputs_v2", "notebooks"}.issubset(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()}
):
    BASE_DIR = BASE_DIR.parent

OUTPUT_BASE = BASE_DIR / "03_outputs_v2" / "IgE_Class_Distributions_Non_Repeat_Patients"
DATA_DIR = OUTPUT_BASE / "Data"
FIG_DIR = OUTPUT_BASE / "Figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "Alpha-gal": BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Data" / "aGal_IgE_20260406_non_repeated_patient_tests_2010_2025.csv",
    "Beef": BASE_DIR / "03_outputs_v2" / "Bos" / "Data" / "Bos_IgE_20260406_non_repeated_patient_tests_2010_2025.csv",
    "Pork": BASE_DIR / "03_outputs_v2" / "Sus" / "Data" / "Sus_IgE_20260406_non_repeated_patient_tests_2010_2025.csv",
    "Lamb_Mutton": BASE_DIR / "03_outputs_v2" / "Ovis" / "Data" / "Ovis_IgE_20260406_non_repeated_patient_tests_2010_2025.csv",
}

DATE_COL = "collected_date"
DOB_COL = "patient_dob_year"
PATIENT_COL = "patient_id"
IGE_COL = "result_for_analysis"

START_DATE = pd.Timestamp("2010-01-01")
END_DATE = pd.Timestamp("2025-12-31")

MIN_VALID_DOB_YEAR = 1900
MAX_VALID_DOB_YEAR = 2025
MIN_VALID_AGE = 0
MAX_VALID_AGE = 120

VALID_YEARS = list(range(2010, 2026))

CLASS_ORDER = ["0", "0/1", "1", "2", "3", "4", "5", "6"]

CLASS_COMMENTS = {
    "0": "Negative",
    "0/1": "Equivocal/Borderline",
    "1": "Low Positive",
    "2": "Moderate Positive",
    "3": "High Positive",
    "4": "Very High Positive",
    "5": "Very High Positive",
    "6": "Very High Positive",
}

CLASS_COLORS = {
    "0": "#d9d9d9",
    "0/1": "#bdbdbd",
    "1": "#c6dbef",
    "2": "#9ecae1",
    "3": "#6baed6",
    "4": "#4292c6",
    "5": "#2171b5",
    "6": "#084594",
}

def clean_name(x):
    return str(x).replace(" ", "_").replace("/", "_")


def classify_ige(x):
    if pd.isna(x):
        return np.nan
    if x < 0.10:
        return "0"
    if x < 0.35:
        return "0/1"
    if x < 0.70:
        return "1"
    if x < 3.50:
        return "2"
    if x < 17.50:
        return "3"
    if x < 50.00:
        return "4"
    if x < 100.00:
        return "5"
    return "6"


def load_and_clean_nonrepeat(file_path, assay_name):
    df = pd.read_csv(file_path, low_memory=False)

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df[IGE_COL] = pd.to_numeric(df[IGE_COL], errors="coerce")
    df[DOB_COL] = pd.to_numeric(df[DOB_COL], errors="coerce")
    df[PATIENT_COL] = df[PATIENT_COL].astype("string").str.strip()

    df["test_year"] = df[DATE_COL].dt.year

    df = df.dropna(subset=[PATIENT_COL, DATE_COL, IGE_COL, DOB_COL]).copy()
    df = df[df[PATIENT_COL] != ""].copy()

    df = df[
        (df[DATE_COL] >= START_DATE)
        & (df[DATE_COL] <= END_DATE)
    ].copy()

    df["test_year"] = df["test_year"].astype(int)
    df["age"] = df["test_year"] - df[DOB_COL]

    df["implausible_dob"] = (
        df[DOB_COL].isna()
        | ~df[DOB_COL].between(MIN_VALID_DOB_YEAR, MAX_VALID_DOB_YEAR, inclusive="both")
        | (df[DOB_COL] > df["test_year"])
        | (df["age"] < MIN_VALID_AGE)
        | (df["age"] > MAX_VALID_AGE)
    )

    implausible_file = DATA_DIR / f"{clean_name(assay_name)}_implausible_dob_rows_excluded.csv"
    df[df["implausible_dob"]].to_csv(implausible_file, index=False)

    df = df[~df["implausible_dob"]].copy()
    df["IgE_class"] = df[IGE_COL].apply(classify_ige)

    cleaned_file = DATA_DIR / f"{clean_name(assay_name)}_nonrepeat_cleaned_valid_dob_2010_2025.csv"
    df.to_csv(cleaned_file, index=False)

    return df


def make_yearly_summary(df, assay_name):
    counts = (
        df.groupby(["IgE_class", "test_year"])[PATIENT_COL]
        .nunique()
        .unstack(fill_value=0)
        .reindex(index=CLASS_ORDER, columns=VALID_YEARS, fill_value=0)
    )

    percentages = counts.div(counts.sum(axis=0), axis=1).fillna(0) * 100

    wide = pd.DataFrame({"Class": CLASS_ORDER})
    for year in VALID_YEARS:
        wide[f"{year}_count"] = counts[year].values
        wide[f"{year}_percentage"] = percentages[year].round(2).values

    wide_file = DATA_DIR / f"{clean_name(assay_name)}_yearly_IgE_class_summary_2010_2025.csv"
    wide.to_csv(wide_file, index=False)

    long = (
        counts.stack()
        .rename("count")
        .reset_index()
        .rename(columns={"test_year": "year"})
    )

    long["total_tests_in_year"] = long["year"].map(counts.sum(axis=0))
    long["percentage"] = (
        long["count"] / long["total_tests_in_year"] * 100
    ).fillna(0).round(2)

    long_file = DATA_DIR / f"{clean_name(assay_name)}_yearly_IgE_class_distribution_long_2010_2025.csv"
    long.to_csv(long_file, index=False)

    return counts, percentages, wide_file, long_file


def plot_yearly_stacked(percentages, assay_name):
    fig, ax = plt.subplots(figsize=(14, 7))

    bottom = np.zeros(len(percentages.columns))

    for cls in CLASS_ORDER:
        vals = percentages.loc[cls].values
        ax.bar(
            [str(y) for y in percentages.columns],
            vals,
            bottom=bottom,
            label=f"Class {cls}",
            color=CLASS_COLORS[cls],
            edgecolor="white",
            linewidth=0.6,
        )
        bottom += vals

    ax.set_title(f"Yearly {assay_name} IgE Class Distribution (2010-2025)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Percentage of tests (%)")
    ax.set_ylim(0, 100)
    ax.legend(title="IgE Class", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=45)
    plt.tight_layout()

    png = FIG_DIR / f"{clean_name(assay_name)}_yearly_IgE_class_distribution_2010_2025.png"
    pdf = FIG_DIR / f"{clean_name(assay_name)}_yearly_IgE_class_distribution_2010_2025.pdf"

    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)

    return png, pdf


def make_overall_class_table_and_plot(df, assay_name):
    class_counts = (
        df["IgE_class"]
        .value_counts(sort=False)
        .reindex(CLASS_ORDER, fill_value=0)
    )

    class_table = pd.DataFrame({
        "Class": CLASS_ORDER,
        "Comment": [CLASS_COMMENTS[c] for c in CLASS_ORDER],
        "Count": class_counts.values,
    })

    class_table["Percent"] = (
        class_table["Count"] / class_table["Count"].sum() * 100
    ).round(2)

    class_csv = DATA_DIR / f"{clean_name(assay_name)}_overall_IgE_class_table_2010_2025.csv"
    class_table.to_csv(class_csv, index=False)

    summary_table = pd.DataFrame({
        "Metric": [
            "Start date",
            "End date",
            "Total non-repeat patient tests after removing implausible DOB rows",
            "Non-repeat patient tests with numeric IgE used for class table",
        ],
        "Value": [
            START_DATE.date().isoformat(),
            END_DATE.date().isoformat(),
            len(df),
            int(class_table["Count"].sum()),
        ],
    })

    summary_csv = DATA_DIR / f"{clean_name(assay_name)}_overall_IgE_summary_2010_2025.csv"
    summary_table.to_csv(summary_csv, index=False)

    sns.set(style="whitegrid")
    fig, ax = plt.subplots(figsize=(10, 6))

    sns.barplot(
        data=class_table,
        x="Class",
        y="Percent",
        color="salmon",
        ax=ax,
    )

    ax.set_title(
        f"Percent in Each ImmunoCAP Class for {assay_name} Non-Repeat Patients "
        f"(2010-2025)\n"
        f"Total tests after removing implausible DOB rows: {len(df):,}"
    )
    ax.set_xlabel("IgE Class")
    ax.set_ylabel("Percent (%)")
    ax.set_ylim(0, max(class_table["Percent"].max() + 5, 10))

    for i, row in class_table.iterrows():
        ax.text(
            i,
            row["Percent"] + 0.3,
            f"{row['Percent']:.1f}%",
            ha="center",
            va="bottom",
        )

    plt.tight_layout()

    png = FIG_DIR / f"{clean_name(assay_name)}_overall_IgE_class_plot_2010_2025.png"
    pdf = FIG_DIR / f"{clean_name(assay_name)}_overall_IgE_class_plot_2010_2025.pdf"

    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)

    return class_csv, summary_csv, png, pdf, class_table


run_summary_rows = []

for assay_name, file_path in FILES.items():
    if not file_path.exists():
        print(f"Skipping {assay_name}: file not found: {file_path}")
        continue

    print(f"\nProcessing {assay_name}...")
    print(f"Input: {file_path}")

    df = load_and_clean_nonrepeat(file_path, assay_name)

    counts, percentages, yearly_wide_csv, yearly_long_csv = make_yearly_summary(
        df,
        assay_name,
    )

    yearly_png, yearly_pdf = plot_yearly_stacked(
        percentages,
        assay_name,
    )

    class_csv, summary_csv, overall_png, overall_pdf, class_table = (
        make_overall_class_table_and_plot(df, assay_name)
    )

    run_summary_rows.append({
        "assay": assay_name,
        "input_file": str(file_path),
        "n_rows_after_valid_date_ige_dob_filters": len(df),
        "n_unique_patients": df[PATIENT_COL].nunique(),
        "yearly_summary_wide_csv": str(yearly_wide_csv),
        "yearly_distribution_long_csv": str(yearly_long_csv),
        "yearly_stacked_png": str(yearly_png),
        "yearly_stacked_pdf": str(yearly_pdf),
        "overall_class_csv": str(class_csv),
        "overall_summary_csv": str(summary_csv),
        "overall_class_png": str(overall_png),
        "overall_class_pdf": str(overall_pdf),
    })

    print(f"Rows after removing implausible DOB rows: {len(df):,}")
    print(f"Unique patients: {df[PATIENT_COL].nunique():,}")
    print(f"Saved yearly summary: {yearly_wide_csv}")
    print(f"Saved yearly long distribution: {yearly_long_csv}")
    print(f"Saved yearly plot: {yearly_png}")
    print(f"Saved overall class table: {class_csv}")
    print(f"Saved overall class plot: {overall_png}")
    print("\nOverall class table:")
    print(class_table.to_string(index=False))

run_summary = pd.DataFrame(run_summary_rows)
run_summary_csv = DATA_DIR / "all_assays_nonrepeat_IgE_class_outputs_summary_2010_2025.csv"
run_summary.to_csv(run_summary_csv, index=False)

print(f"\nSaved combined run summary: {run_summary_csv}")
print(run_summary.to_string(index=False))

# Seasonal analysis of alpha-gal (Figure 2)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2
from scipy.special import expit
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import build_design_matrices

BASE_DIR = Path.cwd().resolve()
while not {"01_raw_data", "02_clean_data", "03_outputs_v2", "notebooks"}.issubset(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()}
):
    BASE_DIR = BASE_DIR.parent

CLEAN_DIR = BASE_DIR / "02_clean_data"
OUTPUT_DIR = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Seasonality_Harmonic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_file = CLEAN_DIR / "aGal_IgE_20260406_clean.csv"

integrated_png = OUTPUT_DIR / "integrated_alpha_gal_seasonality_plot.png"
integrated_pdf = OUTPUT_DIR / "integrated_alpha_gal_seasonality_plot.pdf"
integrated_csv = OUTPUT_DIR / "integrated_alpha_gal_seasonality_plot_data.csv"
harmonic_csv = OUTPUT_DIR / "integrated_alpha_gal_harmonic_results.csv"
implausible_file = OUTPUT_DIR / "integrated_alpha_gal_implausible_dob_rows_excluded.csv"

DATE_COL = "collected_date"
DOB_COL = "patient_dob_year"
PATIENT_COL = "patient_id"
IGE_COL = "result_for_analysis"

MIN_YEAR = 2010
MAX_YEAR = 2025
ADJUSTED_MIN_YEAR = 2013

MIN_VALID_DOB_YEAR = 1900
MAX_VALID_DOB_YEAR = 2025
MIN_VALID_AGE = 0
MAX_VALID_AGE = 120

POSITIVE_CUTOFF = 0.35

MONTH_ORDER = list(range(1, 13))
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

SEVERITY_ORDER = [
    "Low (0.35–0.69 kU/L)",
    "Moderate (0.70–3.49 kU/L)",
    "High (≥3.50 kU/L)",
]

COLORS = {
    "Low (0.35–0.69 kU/L)": "#9ecae1",
    "Moderate (0.70–3.49 kU/L)": "#fdbf6f",
    "High (≥3.50 kU/L)": "#ec8373",
}

N_SIM = 2000
RANDOM_SEED = 12345

FONT_FAMILY = "Times New Roman"       
AXIS_LINEWIDTH = 2
LABEL_PAD = 12
TICK_PAD = 6
RIGHT_AXIS_COLOR = "#0B1AEC"


def format_p_value(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "< 0.001"
    return f"= {p:.3f}"


def likelihood_ratio_test(full_model, null_model):
    lr_stat = 2 * (full_model.llf - null_model.llf)
    df_diff = int(full_model.df_model - null_model.df_model)
    p_value = chi2.sf(lr_stat, df_diff)
    return lr_stat, df_diff, p_value


def severity_group(x):
    if 0.35 <= x < 0.70:
        return "Low (0.35–0.69 kU/L)"
    if 0.70 <= x < 3.50:
        return "Moderate (0.70–3.49 kU/L)"
    if x >= 3.50:
        return "High (≥3.50 kU/L)"
    return np.nan


def marginal_predictions_with_ci(model, prediction_grid, group_col="month_num", n_sim=2000):
    design_info = model.model.data.design_info
    X = build_design_matrices([design_info], prediction_grid, return_type="dataframe")[0]

    beta = model.params.values
    cov = model.cov_params().values

    pred = expit(np.asarray(X) @ beta)
    prediction_grid = prediction_grid.copy()
    prediction_grid["predicted_probability"] = pred

    mean_pred = (
        prediction_grid
        .groupby(group_col)["predicted_probability"]
        .mean()
        .reset_index()
    )

    rng = np.random.default_rng(RANDOM_SEED)
    beta_draws = rng.multivariate_normal(beta, cov, size=n_sim)

    sim_rows = []
    X_np = np.asarray(X)

    for draw in beta_draws:
        tmp = prediction_grid[[group_col]].copy()
        tmp["p"] = expit(X_np @ draw)
        sim_rows.append(tmp.groupby(group_col)["p"].mean().values)

    sim = np.vstack(sim_rows)

    mean_pred["ci_lower"] = np.percentile(sim, 2.5, axis=0)
    mean_pred["ci_upper"] = np.percentile(sim, 97.5, axis=0)

    mean_pred["predicted_rate_percent"] = mean_pred["predicted_probability"] * 100
    mean_pred["ci_lower_percent"] = mean_pred["ci_lower"] * 100
    mean_pred["ci_upper_percent"] = mean_pred["ci_upper"] * 100

    return mean_pred


df = pd.read_csv(input_file, low_memory=False)

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df["collection_year"] = df[DATE_COL].dt.year
df["month_num"] = df[DATE_COL].dt.month
df["month"] = df["month_num"].map(dict(zip(MONTH_ORDER, MONTH_LABELS)))

df[DOB_COL] = pd.to_numeric(df[DOB_COL], errors="coerce")
df[IGE_COL] = pd.to_numeric(df[IGE_COL], errors="coerce")
df[PATIENT_COL] = df[PATIENT_COL].astype("string").str.strip()

df = df.dropna(subset=[PATIENT_COL, DATE_COL, "collection_year", "month_num", IGE_COL]).copy()
df = df[df[PATIENT_COL] != ""].copy()
df["collection_year"] = df["collection_year"].astype(int)
df = df[df["collection_year"].between(MIN_YEAR, MAX_YEAR)].copy()

df["age"] = df["collection_year"] - df[DOB_COL]

df["implausible_dob"] = (
    df[DOB_COL].isna()
    | ~df[DOB_COL].between(MIN_VALID_DOB_YEAR, MAX_VALID_DOB_YEAR, inclusive="both")
    | (df[DOB_COL] > df["collection_year"])
    | (df["age"] < MIN_VALID_AGE)
    | (df["age"] > MAX_VALID_AGE)
)

df[df["implausible_dob"]].to_csv(implausible_file, index=False)

patient_implausible = df.groupby(PATIENT_COL)["implausible_dob"].any()
valid_patient_ids = patient_implausible[~patient_implausible].index
df = df[df[PATIENT_COL].isin(valid_patient_ids)].copy()

tests_per_patient = df.groupby(PATIENT_COL).size()
single_test_patient_ids = tests_per_patient[tests_per_patient == 1].index
single = df[df[PATIENT_COL].isin(single_test_patient_ids)].copy()

single["alpha_gal_positive"] = (single[IGE_COL] >= POSITIVE_CUTOFF).astype(int)
single["sin_month"] = np.sin(2 * np.pi * single["month_num"] / 12)
single["cos_month"] = np.cos(2 * np.pi * single["month_num"] / 12)

monthly_rates = (
    single.groupby("month_num")
    .agg(
        total_tested=(PATIENT_COL, "size"),
        positive_count=("alpha_gal_positive", "sum"),
    )
    .reindex(MONTH_ORDER)
    .fillna(0)
    .astype(int)
    .reset_index()
)

monthly_rates["month"] = monthly_rates["month_num"].map(dict(zip(MONTH_ORDER, MONTH_LABELS)))
monthly_rates["observed_positivity_rate_percent"] = (
    monthly_rates["positive_count"] / monthly_rates["total_tested"] * 100
).replace([np.inf, -np.inf], np.nan).fillna(0)

positive = single[single["alpha_gal_positive"] == 1].copy()
positive["severity"] = positive[IGE_COL].apply(severity_group)

monthly_counts = (
    positive.groupby(["collection_year", "month_num", "severity"])
    .size()
    .reset_index(name="n")
)

years = list(range(MIN_YEAR, MAX_YEAR + 1))

full_grid = (
    pd.MultiIndex.from_product(
        [years, MONTH_ORDER, SEVERITY_ORDER],
        names=["collection_year", "month_num", "severity"],
    )
    .to_frame(index=False)
)

monthly_counts = full_grid.merge(
    monthly_counts,
    on=["collection_year", "month_num", "severity"],
    how="left",
)

monthly_counts["n"] = monthly_counts["n"].fillna(0)

avg_monthly = (
    monthly_counts
    .groupby(["month_num", "severity"], as_index=False)["n"]
    .mean()
)

avg_df = (
    avg_monthly
    .pivot(index="month_num", columns="severity", values="n")
    .reindex(MONTH_ORDER)
    .fillna(0)
)

for col in SEVERITY_ORDER:
    if col not in avg_df.columns:
        avg_df[col] = 0

avg_df = avg_df[SEVERITY_ORDER]
avg_df["month"] = MONTH_LABELS
avg_df["average_positive_samples_per_year"] = avg_df[SEVERITY_ORDER].sum(axis=1)

adjusted = single[single["collection_year"].between(ADJUSTED_MIN_YEAR, MAX_YEAR)].copy()

adjusted_full = smf.glm(
    "alpha_gal_positive ~ sin_month + cos_month + C(collection_year)",
    data=adjusted,
    family=sm.families.Binomial(),
).fit()

adjusted_null = smf.glm(
    "alpha_gal_positive ~ C(collection_year)",
    data=adjusted,
    family=sm.families.Binomial(),
).fit()

adj_lr, adj_df, adj_p = likelihood_ratio_test(adjusted_full, adjusted_null)

harmonic_results = pd.DataFrame([
    {
        "model": "Year-adjusted harmonic model, 2013-2025",
        "term": "sin_month",
        "OR": np.exp(adjusted_full.params["sin_month"]),
        "CI_lower": np.exp(adjusted_full.conf_int().loc["sin_month", 0]),
        "CI_upper": np.exp(adjusted_full.conf_int().loc["sin_month", 1]),
        "p_value": adjusted_full.pvalues["sin_month"],
        "p_value_formatted": format_p_value(adjusted_full.pvalues["sin_month"]),
    },
    {
        "model": "Year-adjusted harmonic model, 2013-2025",
        "term": "cos_month",
        "OR": np.exp(adjusted_full.params["cos_month"]),
        "CI_lower": np.exp(adjusted_full.conf_int().loc["cos_month", 0]),
        "CI_upper": np.exp(adjusted_full.conf_int().loc["cos_month", 1]),
        "p_value": adjusted_full.pvalues["cos_month"],
        "p_value_formatted": format_p_value(adjusted_full.pvalues["cos_month"]),
    },
    {
        "model": "Year-adjusted harmonic model, 2013-2025",
        "term": "overall seasonality LRT",
        "OR": "",
        "CI_lower": "",
        "CI_upper": "",
        "p_value": adj_p,
        "p_value_formatted": format_p_value(adj_p),
        "LR_statistic": adj_lr,
        "df": adj_df,
    },
])

harmonic_results.to_csv(harmonic_csv, index=False)

years_for_adjustment = sorted(adjusted["collection_year"].unique())

pred_grid = pd.DataFrame(
    [
        {"month_num": m, "collection_year": y}
        for m in MONTH_ORDER
        for y in years_for_adjustment
    ]
)

pred_grid["sin_month"] = np.sin(2 * np.pi * pred_grid["month_num"] / 12)
pred_grid["cos_month"] = np.cos(2 * np.pi * pred_grid["month_num"] / 12)

predicted = marginal_predictions_with_ci(
    adjusted_full,
    pred_grid,
    group_col="month_num",
    n_sim=N_SIM,
)

predicted["month"] = predicted["month_num"].map(dict(zip(MONTH_ORDER, MONTH_LABELS)))

plot_data = (
    avg_df.reset_index()
    .merge(monthly_rates, on="month_num", how="left")
    .merge(
        predicted[
            [
                "month_num",
                "predicted_rate_percent",
                "ci_lower_percent",
                "ci_upper_percent",
            ]
        ],
        on="month_num",
        how="left",
    )
)

plot_data.to_csv(integrated_csv, index=False)

plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": AXIS_LINEWIDTH,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 17,
    "legend.fontsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
})

x = np.arange(12)

fig, ax1 = plt.subplots(figsize=(12, 6.4))

bottom = np.zeros(12)

for severity in SEVERITY_ORDER:
    ax1.bar(
        x,
        plot_data[severity],
        bottom=bottom,
        color=COLORS[severity],
        edgecolor="white",
        linewidth=0.7,
        label=severity,
        zorder=2,
    )
    bottom += plot_data[severity].values

bar_totals = plot_data[SEVERITY_ORDER].sum(axis=1)

for i, total in enumerate(bar_totals):
    ax1.text(
        x[i],
        total + 8,
        f"{total:.0f}",
        ha="center",
        va="bottom",
        fontsize=13,
        fontweight="bold",
        color="black",
        zorder=6,
    )
ax1.set_ylabel(
    "Monthly average positive patients",
    fontweight="bold",
    labelpad=LABEL_PAD,
)
ax1.set_xticks(x)
ax1.set_xticklabels(MONTH_LABELS, fontweight="bold")
ax1.set_xlabel(
    "Month",
    fontweight="bold",
    labelpad=LABEL_PAD,
)

ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

for spine in ["bottom", "left"]:
    ax1.spines[spine].set_linewidth(AXIS_LINEWIDTH)

ax1.tick_params(
    axis="both",
    width=AXIS_LINEWIDTH,
    length=5,
    pad=TICK_PAD,
)

for label in ax1.get_xticklabels() + ax1.get_yticklabels():
    label.set_fontweight("bold")

ax1.set_axisbelow(True)
ax1.grid(
    axis="y",
    color="#e6e6e6",
    linestyle="",
    linewidth=0.7,
    alpha=0.8,
)
ax1.grid(axis="x", visible=False)

ax2 = ax1.twinx()

ax2.fill_between(
    x,
    plot_data["ci_lower_percent"],
    plot_data["ci_upper_percent"],
    color=RIGHT_AXIS_COLOR,
    alpha=0.13,
    linewidth=0,
    label="Harmonic 95% CI",
    zorder=1,
)

ax2.plot(
    x,
    plot_data["observed_positivity_rate_percent"],
    color=RIGHT_AXIS_COLOR,
    marker="o",
    linewidth=0,
    markersize=5.5,
    label="Observed positivity rate",
    zorder=4,
)

ax2.plot(
    x,
    plot_data["predicted_rate_percent"],
    color=RIGHT_AXIS_COLOR,
    linestyle="--",
    linewidth=2.4,
    label="Harmonic fitted rate",
    zorder=5,
)

ax2.set_ylabel(
    "Monthly positivity rate (%)",
    color=RIGHT_AXIS_COLOR,
    fontweight="bold",
    labelpad=LABEL_PAD,
)

ax2.spines["top"].set_visible(False)
ax2.spines["left"].set_visible(False)
ax2.spines["right"].set_visible(True)
ax2.spines["right"].set_linewidth(AXIS_LINEWIDTH)
ax2.spines["right"].set_color(RIGHT_AXIS_COLOR)

ax2.tick_params(
    axis="y",
    colors=RIGHT_AXIS_COLOR,
    width=AXIS_LINEWIDTH,
    length=5,
    pad=TICK_PAD,
)

for label in ax2.get_yticklabels():
    label.set_fontweight("bold")
    label.set_color(RIGHT_AXIS_COLOR)

ax2.grid(False)

y2_max = max(
    plot_data["observed_positivity_rate_percent"].max(),
    plot_data["predicted_rate_percent"].max(),
    plot_data["ci_upper_percent"].max(),
)

ax2.set_ylim(
    max(
        0,
        min(
            plot_data["observed_positivity_rate_percent"].min(),
            plot_data["predicted_rate_percent"].min(),
        ) - 6,
    ),
    y2_max + 6,
)

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

legend = ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper left",
    bbox_to_anchor=(0.03, 1.00),
    frameon=False,
    facecolor="white",
    edgecolor="#dddddd",
    framealpha=0.95,
)

for text in legend.get_texts():
    text.set_fontweight("bold")

ax1.set_title("")

fig.patch.set_facecolor("white")
ax1.set_facecolor("white")

plt.tight_layout()

plt.savefig(integrated_png, dpi=1200, bbox_inches="tight")
plt.savefig(integrated_pdf, dpi=1200, bbox_inches="tight")
plt.show()

print(f"Saved integrated plot data: {integrated_csv}")
print(f"Saved harmonic results: {harmonic_csv}")
print(f"Saved integrated PNG: {integrated_png}")
print(f"Saved integrated PDF: {integrated_pdf}")
print(f"Saved implausible DOB rows: {implausible_file}")
print(f"Overall adjusted harmonic seasonality P-value: {format_p_value(adj_p)}")

# Association between alpha-gal and mammalian meat sIgE (Figure 3)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import spearmanr

BASE_DIR = Path.cwd().resolve()

while not {"01_raw_data", "02_clean_data", "03_outputs_v2", "notebooks"}.issubset(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()}
):
    BASE_DIR = BASE_DIR.parent

CLEAN_DIR = BASE_DIR / "02_clean_data"
OUTPUT_DIR = BASE_DIR / "03_outputs_v2" / "Spearman_Correlations" / "Single_Test_Hexbin"
DATA_DIR = OUTPUT_DIR / "Data"
FIG_DIR = OUTPUT_DIR / "Figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "Alpha-gal": CLEAN_DIR / "aGal_IgE_20260406_clean.csv",
    "Bos": CLEAN_DIR / "Bos_IgE_20260406_clean.csv",
    "Sus": CLEAN_DIR / "Sus_IgE_20260406_clean.csv",
    "Ovis": CLEAN_DIR / "Ovis_IgE_20260406_clean.csv",
}

COMPARISONS = [
    ("Alpha-gal", "Bos", "A"),
    ("Alpha-gal", "Sus", "B"),
    ("Alpha-gal", "Ovis", "C"),
]

DISPLAY_NAMES = {
    "Alpha-gal": "Alpha-gal",
    "Bos": "Beef",
    "Sus": "Pork",
    "Ovis": "Lamb/Mutton",
}

DATE_COL = "collected_date"
DOB_COL = "patient_dob_year"
PATIENT_COL = "patient_id"
IGE_COL = "result_for_analysis"

MIN_YEAR = 2010
MAX_YEAR = 2025

MIN_VALID_DOB_YEAR = 1900
MAX_VALID_DOB_YEAR = 2025
MIN_VALID_AGE = 0
MAX_VALID_AGE = 120

POSITIVE_CUTOFF = 0.35

STYLE = {
    "font_family": "Times New Roman",
    "font_size": 8,
    "title_size": 9,
    "axis_label_size": 8.5,
    "tick_label_size": 7.5,
    "panel_label_size": 11,
    "annotation_size": 7,
    "axis_linewidth": 1.0,
    "tick_width": 0.9,
    "tick_length": 3.5,
    "label_pad": 5,
    "title_pad": 5,
    "grid_alpha": 0.10,
    "grid_linewidth": 0.4,
    "cbar_label_pad": 5,
}

X_AXIS_LABEL = r"$\mathbf{log}_{\mathbf{10}}$(Alpha-gal sIgE)"

Y_AXIS_LABELS = {
    "Bos": r"$\mathbf{log}_{\mathbf{10}}$(Beef sIgE)",
    "Sus": r"$\mathbf{log}_{\mathbf{10}}$(Pork sIgE)",
    "Ovis": r"$\mathbf{log}_{\mathbf{10}}$(Lamb/Mutton sIgE)",
}

def clean_name(name):
    return name.replace(" ", "_").replace("/", "_")


def format_p_value(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "< 0.001"
    return f"= {p:.3f}"


def truncate_colormap(cmap_name, minval=0.22, maxval=1.0, n=256):
    cmap = plt.get_cmap(cmap_name)
    return LinearSegmentedColormap.from_list(
        f"truncated_{cmap_name}",
        cmap(np.linspace(minval, maxval, n)),
    )


def load_clean_assay(file_path, assay_name):
    df = pd.read_csv(file_path, low_memory=False)

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df["year"] = df[DATE_COL].dt.year
    df["collected_day"] = df[DATE_COL].dt.normalize()

    df[DOB_COL] = pd.to_numeric(df[DOB_COL], errors="coerce")
    df[IGE_COL] = pd.to_numeric(df[IGE_COL], errors="coerce")
    df[PATIENT_COL] = df[PATIENT_COL].astype("string").str.strip()

    df = df.dropna(subset=[PATIENT_COL, "collected_day", "year", IGE_COL]).copy()
    df = df[df[PATIENT_COL] != ""].copy()

    df["year"] = df["year"].astype(int)
    df = df[df["year"].between(MIN_YEAR, MAX_YEAR)].copy()

    df["age"] = df["year"] - df[DOB_COL]

    df["implausible_dob"] = (
        df[DOB_COL].isna()
        | ~df[DOB_COL].between(MIN_VALID_DOB_YEAR, MAX_VALID_DOB_YEAR, inclusive="both")
        | (df[DOB_COL] > df["year"])
        | (df["age"] < MIN_VALID_AGE)
        | (df["age"] > MAX_VALID_AGE)
    )

    implausible_file = DATA_DIR / f"{clean_name(assay_name)}_implausible_dob_rows_excluded.csv"
    df[df["implausible_dob"]].to_csv(implausible_file, index=False)

    patient_implausible = df.groupby(PATIENT_COL)["implausible_dob"].any()
    valid_patient_ids = patient_implausible[~patient_implausible].index
    df = df[df[PATIENT_COL].isin(valid_patient_ids)].copy()

    test_counts = (
        df.groupby(PATIENT_COL)
        .size()
        .rename(f"{assay_name}_n_tests_2010_2025")
        .reset_index()
    )

    day_level = (
        df.groupby([PATIENT_COL, "collected_day"], as_index=False)
        .agg(
            result_for_analysis=(IGE_COL, "median"),
            n_records_same_day=(IGE_COL, "size"),
            year=("year", "first"),
        )
    )

    day_level = day_level.merge(test_counts, on=PATIENT_COL, how="left")
    day_level["assay"] = assay_name

    return day_level


def make_pair_data(left_assay, right_assay, assay_data):
    left = assay_data[left_assay].copy()
    right = assay_data[right_assay].copy()

    left = left.rename(
        columns={
            "result_for_analysis": f"{left_assay}_ige",
            "n_records_same_day": f"{left_assay}_records_same_day",
        }
    )

    right = right.rename(
        columns={
            "result_for_analysis": f"{right_assay}_ige",
            "n_records_same_day": f"{right_assay}_records_same_day",
        }
    )

    pair = left.merge(
        right,
        on=[PATIENT_COL, "collected_day"],
        how="inner",
        suffixes=("", "_right"),
    )

    return pair[
        [
            PATIENT_COL,
            "collected_day",
            f"{left_assay}_ige",
            f"{right_assay}_ige",
            f"{left_assay}_n_tests_2010_2025",
            f"{right_assay}_n_tests_2010_2025",
        ]
    ].copy()


def get_single_test_pair(pair, left_assay, right_assay):
    left_ige = f"{left_assay}_ige"
    right_ige = f"{right_assay}_ige"
    left_n = f"{left_assay}_n_tests_2010_2025"
    right_n = f"{right_assay}_n_tests_2010_2025"

    out = pair[(pair[left_n] == 1) & (pair[right_n] == 1)].copy()
    out = out.dropna(subset=[left_ige, right_ige]).copy()

    out[f"log10_{clean_name(left_assay)}"] = np.log10(out[left_ige])
    out[f"log10_{clean_name(right_assay)}"] = np.log10(out[right_ige])

    return out


assay_data = {
    assay: load_clean_assay(path, assay)
    for assay, path in FILES.items()
}

plot_datasets = []
summary_rows = []

for left_assay, right_assay, panel_label in COMPARISONS:
    pair = make_pair_data(left_assay, right_assay, assay_data)
    single = get_single_test_pair(pair, left_assay, right_assay)

    left_ige = f"{left_assay}_ige"
    right_ige = f"{right_assay}_ige"

    rho, p_value = spearmanr(single[left_ige], single[right_ige])

    comparison_label = f"{DISPLAY_NAMES[left_assay]} vs {DISPLAY_NAMES[right_assay]}"
    output_stem = clean_name(f"{left_assay} vs {right_assay}")

    single_file = DATA_DIR / f"{output_stem}_single_test_plot_data.csv"
    single.to_csv(single_file, index=False)

    plot_datasets.append({
        "data": single,
        "left_assay": left_assay,
        "right_assay": right_assay,
        "panel_label": panel_label,
        "rho": rho,
        "p_value": p_value,
        "n": len(single),
        "unique_patients": single[PATIENT_COL].nunique(),
    })

    summary_rows.append({
        "panel": panel_label,
        "comparison": comparison_label,
        "spearman_rho": rho,
        "p_value": p_value,
        "p_value_formatted": format_p_value(p_value),
        "n_points": len(single),
        "unique_patients": single[PATIENT_COL].nunique(),
        "plot_data_file": str(single_file),
    })

summary = pd.DataFrame(summary_rows)
summary_file = DATA_DIR / "single_test_spearman_hexbin_summary.csv"
summary.to_csv(summary_file, index=False)

all_x = []
all_y = []

for item in plot_datasets:
    left_log = f"log10_{clean_name(item['left_assay'])}"
    right_log = f"log10_{clean_name(item['right_assay'])}"

    all_x.append(item["data"][left_log])
    all_y.append(item["data"][right_log])

all_x = pd.concat(all_x)
all_y = pd.concat(all_y)

pad = 0.08
xlim = (all_x.min() - pad, all_x.max() + pad)
ylim = (all_y.min() - pad, all_y.max() + pad)

positive_cutoff_log = np.log10(POSITIVE_CUTOFF)

DARK_BLUES = truncate_colormap("Blues", minval=0.28, maxval=1.0)

plt.rcParams.update({
    "font.family": STYLE["font_family"],
    "mathtext.fontset": "custom",
    "mathtext.rm": STYLE["font_family"],
    "mathtext.it": STYLE["font_family"] + ":italic",
    "mathtext.bf": STYLE["font_family"] + ":bold",
    "font.size": STYLE["font_size"],
    "axes.titlesize": STYLE["title_size"],
    "axes.labelsize": STYLE["axis_label_size"],
    "xtick.labelsize": STYLE["tick_label_size"],
    "ytick.labelsize": STYLE["tick_label_size"],
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.weight": "bold",
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
})

fig, axes = plt.subplots(
    3,
    1,
    figsize=(7.2, 10.6),
    sharex=True,
    sharey=True,
    constrained_layout=False,
)

last_hb = None

for i, (ax, item) in enumerate(zip(axes, plot_datasets)):
    data = item["data"]
    left_assay = item["left_assay"]
    right_assay = item["right_assay"]

    left_log = f"log10_{clean_name(left_assay)}"
    right_log = f"log10_{clean_name(right_assay)}"

    hb = ax.hexbin(
        data[left_log],
        data[right_log],
        gridsize=70,
        bins="log",
        cmap=DARK_BLUES,
        mincnt=1,
        linewidths=0,
        alpha=1.0,
        vmin=1,
    )

    last_hb = hb

    ax.axvline(
        positive_cutoff_log,
        color="red",
        linestyle="--",
        linewidth=1.5,
        alpha=0.9,
    )

    ax.axhline(
        positive_cutoff_log,
        color="red",
        linestyle="--",
        linewidth=1.5,
        alpha=0.9,
    )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_box_aspect(1)

    ax.set_title(
        f"{DISPLAY_NAMES[left_assay]} vs. {DISPLAY_NAMES[right_assay]}",
        pad=STYLE["title_pad"],
    )

    if i == len(axes) - 1:
        ax.set_xlabel(X_AXIS_LABEL, labelpad=STYLE["label_pad"])
    else:
        ax.set_xlabel("")
        
    ax.tick_params(labelbottom=True)

    ax.set_ylabel(Y_AXIS_LABELS[right_assay], labelpad=STYLE["label_pad"])

    ax.text(
        -0.09,
        1.14,
        item["panel_label"],
        transform=ax.transAxes,
        fontsize=STYLE["panel_label_size"],
        fontweight="bold",
        va="top",
        ha="left",
    )

    annotation = (
        f"Spearman \u03C1 = {item['rho']:.3f} "
        f"(n = {item['n']:,}; "
        f"$\\it{{P}}$ {format_p_value(item['p_value'])})"
        
    )

    ax.text(
        0.03,
        0.96,
        annotation,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=STYLE["annotation_size"],
        color="black",
        bbox=dict(
            facecolor="none",
            edgecolor="none",
            linewidth=0,
            alpha=0,
            )
    )

    ax.grid(False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(STYLE["axis_linewidth"])
    ax.spines["bottom"].set_linewidth(STYLE["axis_linewidth"])

    ax.tick_params(
        axis="both",
        width=STYLE["tick_width"],
        length=STYLE["tick_length"],
    )

fig.subplots_adjust(
    left=0.14,
    right=0.96,
    bottom=0.075,
    top=0.875,
    hspace=0.24,
)

fig.canvas.draw()
axis_box = axes[0].get_position()

cbar_height = 0.02
cbar_bottom = 0.925

cbar_ax = fig.add_axes([
    axis_box.x0,
    cbar_bottom,
    axis_box.width,
    cbar_height,
])

cbar = fig.colorbar(last_hb, cax=cbar_ax, orientation="horizontal")

cbar.set_label(
    "Paired observation density (log scale)",
    labelpad=STYLE["cbar_label_pad"],
)

cbar.ax.tick_params(
    labelsize=STYLE["tick_label_size"],
    width=STYLE["tick_width"],
    length=STYLE["tick_length"],
)

cbar.ax.xaxis.set_label_position("top")
cbar.ax.xaxis.tick_top()

for spine in cbar.ax.spines.values():
    spine.set_linewidth(STYLE["axis_linewidth"])

for ax in axes:
    ax.title.set_fontweight("bold")
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontweight("bold")

    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")

cbar.ax.xaxis.label.set_fontweight("bold")
for label in cbar.ax.get_xticklabels():
    label.set_fontweight("bold")

plt.show()

panel_png = FIG_DIR / "single_test_spearman_hexbin_panel_3abc.png"
panel_pdf = FIG_DIR / "single_test_spearman_hexbin_panel_3abc.pdf"

fig.savefig(panel_png, dpi=1200, bbox_inches="tight")
fig.savefig(panel_pdf, dpi=1200, bbox_inches="tight")
plt.close(fig)

print(summary)
print(f"\nSaved summary CSV: {summary_file}")
print(f"Saved panel PNG: {panel_png}")
print(f"Saved panel PDF: {panel_pdf}")
print(f"Saved plot data CSVs to: {DATA_DIR}")

# Longitudinal alpha-gal sIgE analysis (Figure 4A-B and supplementary tables)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().resolve()

while not {"01_raw_data", "02_clean_data", "03_outputs_v2", "notebooks"}.issubset(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()}
):
    BASE_DIR = BASE_DIR.parent

CLEAN_DIR = BASE_DIR / "02_clean_data"
OUTPUT_DIR = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "First_Last_Transition_Sankey"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
DATA_DIR = OUTPUT_DIR / "data"

for d in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

input_file = CLEAN_DIR / "aGal_IgE_20260406_clean.csv"

transition_table_file = TABLE_DIR / "aGal_IgE_retested_patient_first_last_status_transition_2010_2025.csv"
patient_level_file = DATA_DIR / "aGal_IgE_retested_patient_first_last_status_patient_level_2010_2025.csv"
implausible_file = DATA_DIR / "aGal_IgE_retested_patient_implausible_dob_rows_2010_2025.csv"

sankey_summary_file = TABLE_DIR / "alpha_gal_sankey_transition_summary.csv"
sankey_png = FIGURE_DIR / "alpha_gal_sankey_transition.png"
sankey_pdf = FIGURE_DIR / "alpha_gal_sankey_transition.pdf"
sankey_html = FIGURE_DIR / "alpha_gal_sankey_transition.html"

DATE_COL = "collected_date"
DOB_COL = "patient_dob_year"
PATIENT_COL = "patient_id"
IGE_COL = "result_for_analysis"

MIN_YEAR = 2010
MAX_YEAR = 2025

MIN_VALID_DOB_YEAR = 1900
MAX_VALID_DOB_YEAR = 2025
MIN_VALID_AGE = 0
MAX_VALID_AGE = 120

STATUS_ORDER = ["Negative", "Borderline", "Positive"]

COLLAPSE_SAME_DAY_TO_MEDIAN = True

def classify_status(x):
    if pd.isna(x):
        return np.nan
    if x < 0.10:
        return "Negative"
    if x < 0.35:
        return "Borderline"
    return "Positive"


def fmt_date(x):
    if pd.isna(x):
        return ""
    return pd.to_datetime(x).date().isoformat()


df = pd.read_csv(input_file, low_memory=False)

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df["collected_day"] = df[DATE_COL].dt.normalize()
df["year"] = df[DATE_COL].dt.year

df[DOB_COL] = pd.to_numeric(df[DOB_COL], errors="coerce")
df[IGE_COL] = pd.to_numeric(df[IGE_COL], errors="coerce")
df[PATIENT_COL] = df[PATIENT_COL].astype("string").str.strip()

df = df.dropna(subset=[PATIENT_COL, DATE_COL, "collected_day", "year", IGE_COL]).copy()
df = df[df[PATIENT_COL] != ""].copy()

df["year"] = df["year"].astype(int)
df = df[df["year"].between(MIN_YEAR, MAX_YEAR)].copy()

df["age"] = df["year"] - df[DOB_COL]

df["implausible_dob"] = (
    df[DOB_COL].isna()
    | ~df[DOB_COL].between(MIN_VALID_DOB_YEAR, MAX_VALID_DOB_YEAR, inclusive="both")
    | (df[DOB_COL] > df["year"])
    | (df["age"] < MIN_VALID_AGE)
    | (df["age"] > MAX_VALID_AGE)
)

df[df["implausible_dob"]].to_csv(implausible_file, index=False)

patient_implausible = df.groupby(PATIENT_COL)["implausible_dob"].any()
valid_patient_ids = patient_implausible[~patient_implausible].index
df = df[df[PATIENT_COL].isin(valid_patient_ids)].copy()

if COLLAPSE_SAME_DAY_TO_MEDIAN:
    analysis = (
        df.groupby([PATIENT_COL, "collected_day"], as_index=False)
        .agg(
            test_date=("collected_day", "first"),
            test_result=(IGE_COL, "median"),
            test_year=("year", "first"),
            n_raw_records_same_day=(IGE_COL, "size"),
        )
    )

    analysis = analysis.sort_values([PATIENT_COL, "test_date"]).copy()
    test_count_col = "n_distinct_test_dates_2010_2025"
    analysis_unit = "patient-day median"

else:
    analysis = df[[PATIENT_COL, DATE_COL, IGE_COL, "year"]].copy()
    analysis = analysis.rename(
        columns={
            DATE_COL: "test_date",
            IGE_COL: "test_result",
            "year": "test_year",
        }
    )
    analysis["n_raw_records_same_day"] = 1
    analysis = analysis.sort_values([PATIENT_COL, "test_date"]).copy()
    test_count_col = "n_tests_2010_2025"
    analysis_unit = "raw test record"

test_counts = analysis.groupby(PATIENT_COL).size()
retested_patient_ids = test_counts[test_counts > 1].index

retested = analysis[analysis[PATIENT_COL].isin(retested_patient_ids)].copy()
retested = retested.sort_values([PATIENT_COL, "test_date"]).copy()

patient_rows = []

for patient_id, sub in retested.groupby(PATIENT_COL, sort=True):
    sub = sub.sort_values("test_date").reset_index(drop=True)

    first = sub.iloc[0]
    last = sub.iloc[-1]

    first_status = classify_status(first["test_result"])
    last_status = classify_status(last["test_result"])

    patient_rows.append({
        "patient_id": patient_id,
        test_count_col: len(sub),
        "analysis_unit": analysis_unit,
        "first_test_date": fmt_date(first["test_date"]),
        "first_test_result": first["test_result"],
        "first_test_status": first_status,
        "last_test_date": fmt_date(last["test_date"]),
        "last_test_result": last["test_result"],
        "last_test_status": last_status,
        "transition": f"{first_status} -> {last_status}",
        "has_any_same_day_duplicate_records": bool(
            sub["n_raw_records_same_day"].gt(1).any()
        ),
        "total_raw_records_represented": int(sub["n_raw_records_same_day"].sum()),
    })

patient_level = pd.DataFrame(patient_rows)
patient_level.to_csv(patient_level_file, index=False)

total_retested = len(patient_level)

transition_counts = (
    patient_level
    .groupby(["first_test_status", "last_test_status"])
    .size()
    .reset_index(name="Count")
)

all_transitions = pd.MultiIndex.from_product(
    [STATUS_ORDER, STATUS_ORDER],
    names=["first_test_status", "last_test_status"],
).to_frame(index=False)

transition_table = all_transitions.merge(
    transition_counts,
    on=["first_test_status", "last_test_status"],
    how="left",
)

transition_table["Count"] = transition_table["Count"].fillna(0).astype(int)
transition_table["Percent of retested patients"] = (
    transition_table["Count"] / total_retested * 100
).fillna(0).round(1)

transition_table["Transition"] = (
    transition_table["first_test_status"]
    + " -> "
    + transition_table["last_test_status"]
)

transition_table = transition_table[
    [
        "Transition",
        "first_test_status",
        "last_test_status",
        "Count",
        "Percent of retested patients",
    ]
].copy()

total_row = pd.DataFrame([{
    "Transition": "Total retested alpha-gal patients",
    "first_test_status": "",
    "last_test_status": "",
    "Count": total_retested,
    "Percent of retested patients": 100.0,
}])

transition_table_with_total = pd.concat(
    [transition_table, total_row],
    ignore_index=True,
)

transition_table_with_total.to_csv(transition_table_file, index=False)
transition_table.to_csv(sankey_summary_file, index=False)

print(f"Analysis unit: {analysis_unit}")
print(f"Saved transition table: {transition_table_file}")
print(f"Saved Sankey summary table: {sankey_summary_file}")
print(f"Saved patient-level table: {patient_level_file}")
print(f"Saved implausible DOB rows: {implausible_file}")
print(f"Total retested alpha-gal patients: {total_retested:,}")
print("\nTransition table:")
print(transition_table_with_total.to_string(index=False))


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import multivariate_logrank_test
except ImportError as e:
    raise ImportError(
        "This script requires lifelines. Install it with: %pip install lifelines"
    ) from e

BASE_DIR = Path.cwd().resolve()

while not {"01_raw_data", "02_clean_data", "03_outputs_v2", "notebooks"}.issubset(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()}
):
    BASE_DIR = BASE_DIR.parent

CLEAN_DIR = BASE_DIR / "02_clean_data"
OUTPUT_DIR = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Kaplan_Meier_Seroreversion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_file = CLEAN_DIR / "aGal_IgE_20260406_clean.csv"

strat_png = OUTPUT_DIR / "alpha_gal_km_seroreversion_by_initial_intensity.png"
strat_pdf = OUTPUT_DIR / "alpha_gal_km_seroreversion_by_initial_intensity.pdf"

summary_csv = OUTPUT_DIR / "kaplan_meier_seroreversion_summary.csv"
strat_summary_csv = OUTPUT_DIR / "kaplan_meier_seroreversion_by_initial_intensity_summary.csv"
patient_level_csv = OUTPUT_DIR / "kaplan_meier_seroreversion_patient_level_data.csv"
risk_table_csv = OUTPUT_DIR / "kaplan_meier_seroreversion_by_initial_intensity_risk_table.csv"
curve_data_csv = OUTPUT_DIR / "kaplan_meier_seroreversion_by_initial_intensity_curve_data.csv"
implausible_csv = OUTPUT_DIR / "kaplan_meier_seroreversion_implausible_dob_rows_excluded.csv"

DATE_COL = "collected_date"
DOB_COL = "patient_dob_year"
PATIENT_COL = "patient_id"
IGE_COL = "result_for_analysis"

MIN_YEAR = 2010
MAX_YEAR = 2025

MIN_VALID_DOB_YEAR = 1900
MAX_VALID_DOB_YEAR = 2025
MIN_VALID_AGE = 0
MAX_VALID_AGE = 120

POSITIVE_CUTOFF = 0.35
TIMEPOINTS = [1, 2, 5, 10]
RISK_TIMES = np.arange(0, 13, 2)

INTENSITY_ORDER = [
    "Low positive (0.35–0.69 kU/L)",
    "Moderate positive (0.70–3.49 kU/L)",
    "High positive (≥3.50 kU/L)",
]

INTENSITY_COLORS = {
    "Low positive (0.35–0.69 kU/L)": "#1f77b4",
    "Moderate positive (0.70–3.49 kU/L)": "#ff7f0e",
    "High positive (≥3.50 kU/L)": "#2ca02c",
}

def initial_intensity(x):
    if 0.35 <= x < 0.70:
        return "Low positive (0.35–0.69 kU/L)"
    if 0.70 <= x < 3.50:
        return "Moderate positive (0.70–3.49 kU/L)"
    if x >= 3.50:
        return "High positive (≥3.50 kU/L)"
    return np.nan


def fmt_p_value(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def fmt_float(x, digits=2):
    if pd.isna(x) or np.isinf(x):
        return "Not estimable"
    return f"{x:.{digits}f}"


def km_summary_row(data, label):
    kmf = KaplanMeierFitter()
    kmf.fit(
        durations=data["follow_up_years"],
        event_observed=data["event_seroreversion"],
        label=label,
    )

    follow = data["follow_up_years"].dropna()
    q1 = follow.quantile(0.25)
    q2 = follow.median()
    q3 = follow.quantile(0.75)

    row = {
        "group": label,
        "N": len(data),
        "events": int(data["event_seroreversion"].sum()),
        "censored": int((1 - data["event_seroreversion"]).sum()),
        "median_follow_up_years": round(q2, 2),
        "IQR_follow_up_years": f"{q1:.2f}-{q3:.2f}",
        "median_time_to_seroreversion_years": fmt_float(kmf.median_survival_time_, 2),
    }

    for t in TIMEPOINTS:
        if follow.max() >= t:
            row[f"prob_remaining_positive_{t}y"] = round(float(kmf.predict(t)), 3)
        else:
            row[f"prob_remaining_positive_{t}y"] = ""

    return row, kmf


df = pd.read_csv(input_file, low_memory=False)

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df["collection_year"] = df[DATE_COL].dt.year
df["collected_day"] = df[DATE_COL].dt.normalize()

df[DOB_COL] = pd.to_numeric(df[DOB_COL], errors="coerce")
df[IGE_COL] = pd.to_numeric(df[IGE_COL], errors="coerce")
df[PATIENT_COL] = df[PATIENT_COL].astype("string").str.strip()

df = df.dropna(subset=[PATIENT_COL, "collected_day", "collection_year", IGE_COL]).copy()
df = df[df[PATIENT_COL] != ""].copy()

df["collection_year"] = df["collection_year"].astype(int)
df = df[df["collection_year"].between(MIN_YEAR, MAX_YEAR)].copy()

df["age"] = df["collection_year"] - df[DOB_COL]

df["implausible_dob"] = (
    df[DOB_COL].isna()
    | ~df[DOB_COL].between(MIN_VALID_DOB_YEAR, MAX_VALID_DOB_YEAR, inclusive="both")
    | (df[DOB_COL] > df["collection_year"])
    | (df["age"] < MIN_VALID_AGE)
    | (df["age"] > MAX_VALID_AGE)
)

df[df["implausible_dob"]].to_csv(implausible_csv, index=False)

patient_implausible = df.groupby(PATIENT_COL)["implausible_dob"].any()
valid_patient_ids = patient_implausible[~patient_implausible].index
df = df[df[PATIENT_COL].isin(valid_patient_ids)].copy()

day_level = (
    df.groupby([PATIENT_COL, "collected_day"], as_index=False)
    .agg(
        alpha_gal_ige=(IGE_COL, "median"),
        collection_year=("collection_year", "first"),
        n_records_same_day=(IGE_COL, "size"),
    )
)

day_level = day_level.sort_values([PATIENT_COL, "collected_day"]).copy()

patient_rows = []

for patient_id, sub in day_level.groupby(PATIENT_COL, sort=True):
    sub = sub.sort_values("collected_day").reset_index(drop=True)

    if len(sub) < 2:
        continue

    first = sub.iloc[0]
    first_ige = float(first["alpha_gal_ige"])

    if first_ige < POSITIVE_CUTOFF:
        continue

    first_date = first["collected_day"]
    later = sub.iloc[1:].copy()

    event_rows = later[later["alpha_gal_ige"] < POSITIVE_CUTOFF]

    if len(event_rows) > 0:
        event_row = event_rows.iloc[0]
        end_date = event_row["collected_day"]
        end_ige = float(event_row["alpha_gal_ige"])
        event = 1
    else:
        event_row = sub.iloc[-1]
        end_date = event_row["collected_day"]
        end_ige = float(event_row["alpha_gal_ige"])
        event = 0

    follow_up_days = (end_date - first_date).days
    follow_up_years = follow_up_days / 365.25

    patient_rows.append({
        "patient_id": patient_id,
        "n_test_dates": len(sub),
        "first_positive_date": first_date,
        "first_alpha_gal_ige": first_ige,
        "initial_intensity": initial_intensity(first_ige),
        "end_date": end_date,
        "end_alpha_gal_ige": end_ige,
        "event_seroreversion": event,
        "follow_up_days": follow_up_days,
        "follow_up_years": follow_up_years,
    })

km_data = pd.DataFrame(patient_rows)
km_data = km_data.dropna(subset=["initial_intensity", "follow_up_years"]).copy()
km_data = km_data[km_data["follow_up_years"] >= 0].copy()

if km_data.empty:
    raise ValueError("No eligible repeat-tested first-positive alpha-gal patients were found.")

km_data.to_csv(patient_level_csv, index=False)

overall_row, overall_kmf = km_summary_row(km_data, "Overall")
overall_summary = pd.DataFrame([overall_row])
overall_summary.to_csv(summary_csv, index=False)

strat_rows = []
strat_kmfs = {}

for_group_order = [g for g in INTENSITY_ORDER if g in set(km_data["initial_intensity"])]

for group in for_group_order:
    group_data = km_data[km_data["initial_intensity"] == group].copy()
    row, kmf = km_summary_row(group_data, group)
    strat_rows.append(row)
    strat_kmfs[group] = kmf

strat_summary = pd.DataFrame(strat_rows)

if len(for_group_order) >= 2:
    lr_result = multivariate_logrank_test(
        km_data["follow_up_years"],
        km_data["initial_intensity"],
        km_data["event_seroreversion"],
    )
    logrank_p = lr_result.p_value
    logrank_stat = lr_result.test_statistic
else:
    logrank_p = np.nan
    logrank_stat = np.nan

strat_summary["log_rank_p_value_overall"] = fmt_p_value(logrank_p)
strat_summary["log_rank_test_statistic"] = (
    round(logrank_stat, 3) if pd.notna(logrank_stat) else ""
)

strat_summary.to_csv(strat_summary_csv, index=False)

risk_rows = []

for group in for_group_order:
    group_data = km_data[km_data["initial_intensity"] == group].copy()

    for t in RISK_TIMES:
        at_risk = int((group_data["follow_up_years"] > t).sum())

        events = int(
            (
                (group_data["event_seroreversion"] == 1)
                & (group_data["follow_up_years"] <= t)
            ).sum()
        )

        censored = int(
            (
                (group_data["event_seroreversion"] == 0)
                & (group_data["follow_up_years"] <= t)
            ).sum()
        )

        risk_rows.append({
            "initial_intensity": group,
            "time_years": t,
            "at_risk": at_risk,
            "events": events,
            "censored": censored,
        })

risk_table = pd.DataFrame(risk_rows)
risk_table.to_csv(risk_table_csv, index=False)

curve_rows = []

for group in for_group_order:
    kmf = strat_kmfs[group]

    survival = kmf.survival_function_.reset_index()
    survival.columns = ["time_years", "survival_probability"]

    ci = kmf.confidence_interval_.reset_index()
    ci.columns = ["time_years", "ci_lower", "ci_upper"]

    curve = survival.merge(ci, on="time_years", how="left")
    curve["initial_intensity"] = group

    curve_rows.append(curve)

curve_data = pd.concat(curve_rows, ignore_index=True)
curve_data = curve_data[
    ["initial_intensity", "time_years", "survival_probability", "ci_lower", "ci_upper"]
].copy()

curve_data.to_csv(curve_data_csv, index=False)


print(f"Saved patient-level survival data: {patient_level_csv}")
print(f"Saved KM curve-coordinate data: {curve_data_csv}")
print(f"Saved separate risk table CSV: {risk_table_csv}")
print(f"Saved overall summary CSV: {summary_csv}")
print(f"Saved stratified summary CSV: {strat_summary_csv}")
print(f"Saved implausible DOB rows: {implausible_csv}")
print("\nOverall summary:")
print(overall_summary.to_string(index=False))
print("\nStratified summary:")
print(strat_summary.to_string(index=False))


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import plotly.graph_objects as go

def crop_bottom(img, crop_fraction=0.18):
    h = img.shape[0]
    crop_pixels = int(h * crop_fraction)
    return img[: h - crop_pixels, :, :]


COMBINED_DIR = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Combined_Figures"
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

panel_a_png = COMBINED_DIR / "panel_A_sankey_final_style.png"
panel_b_png = COMBINED_DIR / "panel_B_km_final_style.png"

combined_png = COMBINED_DIR / "alpha_gal_sankey_km_panel_AB_final.png"
combined_pdf = COMBINED_DIR / "alpha_gal_sankey_km_panel_AB_final.pdf"

FONT_FAMILY = "Times New Roman"

PANEL_WIDTH_PX = 1100
PANEL_HEIGHT_PX = 680
EXPORT_SCALE = 6


node_labels = ["", "", "", "", "", ""]

node_map = {
    "First: Negative": 0,
    "First: Borderline": 1,
    "First: Positive": 2,
    "Last: Negative": 3,
    "Last: Borderline": 4,
    "Last: Positive": 5,
}

status_colors = {
    "Negative": "rgba(120, 120, 120, 0.85)",
    "Borderline": "rgba(245, 166, 35, 0.85)",
    "Positive": "rgba(200, 45, 45, 0.85)",
}

link_colors = {
    "Negative": "rgba(120, 120, 120, 0.35)",
    "Borderline": "rgba(245, 166, 35, 0.35)",
    "Positive": "rgba(200, 45, 45, 0.35)",
}

label_text_colors = {
    "Negative": "rgb(60, 60, 60)",
    "Borderline": "rgb(220, 120, 20)",
    "Positive": "rgb(200, 0, 0)",
}

sources = []
targets = []
values = []
colors = []
customdata = []

for _, row in transition_table.iterrows():
    first = row["first_test_status"]
    last = row["last_test_status"]
    count = int(row["Count"])
    pct = row["Percent of retested patients"]

    if count == 0:
        continue

    sources.append(node_map[f"First: {first}"])
    targets.append(node_map[f"Last: {last}"])
    values.append(count)
    colors.append(link_colors[first])
    customdata.append(f"{first} -> {last}<br>{count:,} patients<br>{pct:.1f}%")

node_colors = [
    status_colors["Negative"],
    status_colors["Borderline"],
    status_colors["Positive"],
    status_colors["Negative"],
    status_colors["Borderline"],
    status_colors["Positive"],
]

initial_totals = (
    transition_table.groupby("first_test_status")["Count"]
    .sum()
    .reindex(STATUS_ORDER)
)

final_totals = (
    transition_table.groupby("last_test_status")["Count"]
    .sum()
    .reindex(STATUS_ORDER)
)

x_positions = [0.02, 0.02, 0.02, 0.98, 0.98, 0.98]
y_positions = [0.12, 0.45, 0.78, 0.12, 0.45, 0.78]

fig_sankey = go.Figure(
    data=[
        go.Sankey(
            arrangement="fixed",
            domain=dict(
                x=[0.08, 0.92],
                y=[0.10, 0.96],
            ),
            node=dict(
                pad=28,
                thickness=22,
                line=dict(color="black", width=0.5),
                label=node_labels,
                color=node_colors,
                x=x_positions,
                y=y_positions,
            ),
            link=dict(
                source=sources,
                target=targets,
                value=values,
                color=colors,
                customdata=customdata,
                hovertemplate="%{customdata}<extra></extra>",
            ),
        )
    ]
)

annotations = [
    dict(
        text="<b>Initial status</b>",
        x=0.02,
        y=1.04,
        xref="paper",
        yref="paper",
        showarrow=False,
        xanchor="center",
        font=dict(size=18, family=FONT_FAMILY, color="black"),
    ),
    dict(
        text="<b>Final status</b>",
        x=0.98,
        y=1.04,
        xref="paper",
        yref="paper",
        showarrow=False,
        xanchor="center",
        font=dict(size=18, family=FONT_FAMILY, color="black"),
    ),
]

block_center_y = {
    "Negative": 0.85,
    "Borderline": 0.572,
    "Positive": 0.32,
}

left_label_x = 0.075
right_label_x = 0.925

left_count_x = 0.101
right_count_x = 0.899

status_ranges = {
    "Negative": "(<0.10 kU/L)",
    "Borderline": "(0.10–0.34 kU/L)",
    "Positive": "(≥0.35 kU/L)",
}
for status in STATUS_ORDER:
    annotations.append(
    dict(
        text=(
            f"<b>{status}</b><br>"
            f"<b>{status_ranges[status]}</b>"
        ),
        x=left_label_x,
        y=block_center_y[status],
        xref="paper",
        yref="paper",
        showarrow=False,
        xanchor="right",
        yanchor="middle",
        align="right",
        font=dict(
            size=16,
            family=FONT_FAMILY,
            color=label_text_colors[status],
            ),
        )
    )

    annotations.append(
        dict(
            text=f"<b>{status}</b>",
            x=right_label_x,
            y=block_center_y[status],
            xref="paper",
            yref="paper",
            showarrow=False,
            xanchor="left",
            yanchor="middle",
            font=dict(
                size=16,
                family=FONT_FAMILY,
                color=label_text_colors[status],
            ),
        )
    )

    annotations.append(
        dict(
            text=f"<b>{int(initial_totals.loc[status]):,}</b>",
            x=left_count_x,
            y=block_center_y[status],
            xref="paper",
            yref="paper",
            showarrow=False,
            xanchor="center",
            yanchor="middle",
            textangle=-90,
            font=dict(
                size=14,
                family=FONT_FAMILY,
                color="white",
            ),
        )
    )

    annotations.append(
        dict(
            text=f"<b>{int(final_totals.loc[status]):,}</b>",
            x=right_count_x,
            y=block_center_y[status],
            xref="paper",
            yref="paper",
            showarrow=False,
            xanchor="center",
            yanchor="middle",
            textangle=-90,
            font=dict(
                size=14,
                family=FONT_FAMILY,
                color="white",
            ),
        )
    )

fig_sankey.update_layout(
    title=dict(text=""),
    annotations=annotations,
    font=dict(size=14, family=FONT_FAMILY, color="black"),
    width=PANEL_WIDTH_PX,
    height=PANEL_HEIGHT_PX,
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=130, r=130, t=90, b=95),
)

fig_sankey.write_image(
    str(panel_a_png),
    width=PANEL_WIDTH_PX,
    height=PANEL_HEIGHT_PX,
    scale=EXPORT_SCALE,
)


KM_DPI = 1200

KM_PLOT_BOX = [0.12, 0.35, 0.72, 0.62]

plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.size": 10,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

fig_km = plt.figure(
    figsize=(11, 6.8),
    dpi=KM_DPI,
    facecolor="white",
)

ax = fig_km.add_axes(KM_PLOT_BOX)

for group in for_group_order:
    kmf = strat_kmfs[group]
    color = INTENSITY_COLORS.get(group, None)

    kmf.plot_survival_function(
        ax=ax,
        ci_show=True,
        color=color,
        linewidth=1.5,
        ci_alpha=0.15,
        show_censors=False,
    )

ax.set_xlabel("Follow-up time (years)", labelpad=8)
ax.set_ylabel("Probability of remaining alpha-gal sIgE positive", labelpad=8)

ax.set_ylim(0, 1.02)
ax.set_xlim(left=0)

ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.9)
ax.spines["bottom"].set_linewidth(0.9)

ax.tick_params(axis="both", width=0.8, length=3)

ax.legend(
    title="",
    frameon=False,
    loc="upper right",
    fontsize=10,
    handlelength=2.4,
)

fig_km.savefig(panel_b_png, dpi=KM_DPI)
plt.close(fig_km)

sankey_img = crop_bottom(mpimg.imread(panel_a_png), crop_fraction=0.18)
km_img = mpimg.imread(panel_b_png)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(7.4, 9.2),
    dpi=1200,
)

axes[0].set_zorder(1)
axes[1].set_zorder(2)
axes[1].patch.set_alpha(0)

panel_label_positions = {
    "A": (0.105, 0.93),
    "B": (0.098, 1.05),
}

for ax_i, img, panel_label in zip(axes, [sankey_img, km_img], ["A", "B"]):
    ax_i.imshow(img)
    ax_i.axis("off")

    x_pos, y_pos = panel_label_positions[panel_label]

    ax_i.text(
        x_pos,
        y_pos,
        panel_label,
        transform=ax_i.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left",
        color="black",
        zorder=10,
    )
plt.subplots_adjust(
    left=0.01,
    right=0.99,
    top=0.99,
    bottom=0.01,
    hspace=0.04,
)
plt.show()

fig.savefig(combined_png, dpi=1200, bbox_inches="tight")
fig.savefig(combined_pdf, dpi=1200, bbox_inches="tight")
plt.close(fig)

print(f"Saved panel A PNG: {panel_a_png}")
print(f"Saved panel B PNG: {panel_b_png}")
print(f"Saved combined PNG: {combined_png}")
print(f"Saved combined PDF: {combined_pdf}")

# Predicted alpha-gal sIgE positivity by age and year (Supplementary figure 1 and supplementary table 1)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2
import statsmodels.api as sm
import statsmodels.formula.api as smf

BASE_DIR = Path.cwd().resolve()
while not {"01_raw_data", "02_clean_data", "03_outputs_v2", "notebooks"}.issubset(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()}
):
    BASE_DIR = BASE_DIR.parent

CLEAN_DIR = BASE_DIR / "02_clean_data"
OUTPUT_DIR = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Age_Year_Interaction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_file = CLEAN_DIR / "aGal_IgE_20260406_clean.csv"

model_summary_csv = OUTPUT_DIR / "age_year_interaction_model_comparison.csv"
predicted_csv = OUTPUT_DIR / "age_year_predicted_positivity_probabilities.csv"
observed_csv = OUTPUT_DIR / "age_year_observed_positivity_rates.csv"
implausible_csv = OUTPUT_DIR / "age_year_interaction_implausible_dob_rows_excluded.csv"
results_txt = OUTPUT_DIR / "age_year_interaction_results_discussion_text.txt"

heatmap_png = OUTPUT_DIR / "age_year_predicted_positivity_heatmap.png"
heatmap_pdf = OUTPUT_DIR / "age_year_predicted_positivity_heatmap.pdf"

lineplot_png = OUTPUT_DIR / "age_year_predicted_positivity_lineplot.png"
lineplot_pdf = OUTPUT_DIR / "age_year_predicted_positivity_lineplot.pdf"

DATE_COL = "collected_date"
DOB_COL = "patient_dob_year"
PATIENT_COL = "patient_id"
IGE_COL = "result_for_analysis"

MIN_YEAR = 2013
MAX_YEAR = 2025

MIN_VALID_DOB_YEAR = 1900
MAX_VALID_DOB_YEAR = 2025
MIN_VALID_AGE = 0
MAX_VALID_AGE = 120

POSITIVE_CUTOFF = 0.35

AGE_BINS = [0, 10, 20, 30, 40, 50, 60, 70, np.inf]
AGE_LABELS = [
    "0–9",
    "10–19",
    "20–29",
    "30–39",
    "40–49",
    "50–59",
    "60–69",
    "70+",
]

YEAR_ORDER = list(range(MIN_YEAR, MAX_YEAR + 1))

def format_p_value(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "< 0.001"
    return f"= {p:.3f}"


def likelihood_ratio_test(full_model, reduced_model):
    lr_stat = 2 * (full_model.llf - reduced_model.llf)
    df_diff = int(full_model.df_model - reduced_model.df_model)
    p_value = chi2.sf(lr_stat, df_diff)
    return lr_stat, df_diff, p_value


df = pd.read_csv(input_file, low_memory=False)

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
df["collection_year"] = df[DATE_COL].dt.year

df[DOB_COL] = pd.to_numeric(df[DOB_COL], errors="coerce")
df[IGE_COL] = pd.to_numeric(df[IGE_COL], errors="coerce")
df[PATIENT_COL] = df[PATIENT_COL].astype("string").str.strip()

df = df.dropna(subset=[PATIENT_COL, DATE_COL, "collection_year", IGE_COL]).copy()
df = df[df[PATIENT_COL] != ""].copy()
df["collection_year"] = df["collection_year"].astype(int)
df = df[df["collection_year"].between(MIN_YEAR, MAX_YEAR)].copy()

df["age"] = df["collection_year"] - df[DOB_COL]

df["implausible_dob"] = (
    df[DOB_COL].isna()
    | ~df[DOB_COL].between(MIN_VALID_DOB_YEAR, MAX_VALID_DOB_YEAR, inclusive="both")
    | (df[DOB_COL] > df["collection_year"])
    | (df["age"] < MIN_VALID_AGE)
    | (df["age"] > MAX_VALID_AGE)
)

df[df["implausible_dob"]].to_csv(implausible_csv, index=False)

patient_implausible = df.groupby(PATIENT_COL)["implausible_dob"].any()
valid_patient_ids = patient_implausible[~patient_implausible].index
df = df[df[PATIENT_COL].isin(valid_patient_ids)].copy()

tests_per_patient = df.groupby(PATIENT_COL).size()
single_patient_ids = tests_per_patient[tests_per_patient == 1].index

single = df[df[PATIENT_COL].isin(single_patient_ids)].copy()

single["positive_binary"] = (single[IGE_COL] >= POSITIVE_CUTOFF).astype(int)

single["age_group"] = pd.cut(
    single["age"],
    bins=AGE_BINS,
    labels=AGE_LABELS,
    right=False,
    include_lowest=True,
)

single = single.dropna(
    subset=["age_group", "collection_year", "positive_binary"]
).copy()

single["age_group"] = pd.Categorical(
    single["age_group"],
    categories=AGE_LABELS,
    ordered=True,
)

single["collection_year"] = pd.Categorical(
    single["collection_year"],
    categories=YEAR_ORDER,
    ordered=True,
)

observed = (
    single
    .groupby(["collection_year", "age_group"], observed=False)
    .agg(
        total_tested=(PATIENT_COL, "size"),
        positive_count=("positive_binary", "sum"),
    )
    .reset_index()
)

observed["observed_positivity_percent"] = (
    observed["positive_count"] / observed["total_tested"] * 100
).replace([np.inf, -np.inf], np.nan)

observed.to_csv(observed_csv, index=False)

model_main = smf.glm(
    "positive_binary ~ C(age_group) + C(collection_year)",
    data=single,
    family=sm.families.Binomial(),
).fit()

model_interaction = smf.glm(
    "positive_binary ~ C(age_group) * C(collection_year)",
    data=single,
    family=sm.families.Binomial(),
).fit()

lr_stat, df_diff, lr_p = likelihood_ratio_test(model_interaction, model_main)

model_comparison = pd.DataFrame(
    [
        {
            "model": "Model 1: age group + collection year",
            "AIC": model_main.aic,
            "log_likelihood": model_main.llf,
            "df_model": model_main.df_model,
        },
        {
            "model": "Model 2: age group * collection year",
            "AIC": model_interaction.aic,
            "log_likelihood": model_interaction.llf,
            "df_model": model_interaction.df_model,
        },
    ]
)

lrt_row = pd.DataFrame(
    [
        {
            "model": "Likelihood ratio test: interaction vs main effects",
            "AIC": "",
            "log_likelihood": "",
            "df_model": "",
            "LR_statistic": lr_stat,
            "df": df_diff,
            "p_value": lr_p,
            "p_value_formatted": format_p_value(lr_p),
        }
    ]
)

model_output = pd.concat([model_comparison, lrt_row], ignore_index=True)
model_output.to_csv(model_summary_csv, index=False)

prediction_grid = pd.DataFrame(
    [
        {"collection_year": year, "age_group": age_group}
        for year in YEAR_ORDER
        for age_group in AGE_LABELS
    ]
)

prediction_grid["age_group"] = pd.Categorical(
    prediction_grid["age_group"],
    categories=AGE_LABELS,
    ordered=True,
)

prediction_grid["collection_year"] = pd.Categorical(
    prediction_grid["collection_year"],
    categories=YEAR_ORDER,
    ordered=True,
)

pred = model_interaction.get_prediction(prediction_grid).summary_frame()

prediction_grid["predicted_probability"] = pred["mean"]
prediction_grid["ci_lower"] = pred["mean_ci_lower"]
prediction_grid["ci_upper"] = pred["mean_ci_upper"]

prediction_grid["predicted_positivity_percent"] = (
    prediction_grid["predicted_probability"] * 100
)
prediction_grid["ci_lower_percent"] = prediction_grid["ci_lower"] * 100
prediction_grid["ci_upper_percent"] = prediction_grid["ci_upper"] * 100

prediction_grid = prediction_grid.merge(
    observed,
    on=["collection_year", "age_group"],
    how="left",
)

for col in [
    "predicted_positivity_percent",
    "ci_lower_percent",
    "ci_upper_percent",
    "observed_positivity_percent",
]:
    prediction_grid[col] = prediction_grid[col].round(1)

prediction_grid.to_csv(predicted_csv, index=False)

plt.rcParams.update(
    {
        "font.family": "Times New Roman",
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 15,
        "legend.fontsize": 12,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "font.weight": "bold",
        "axes.labelweight": "bold",
    }
)

heatmap_data = prediction_grid.pivot(
    index="age_group",
    columns="collection_year",
    values="predicted_positivity_percent",
)

fig, ax = plt.subplots(figsize=(10.5, 5.5))

sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap="viridis_r",
    annot=True,
    fmt=".1f",
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Predicted positivity (%)"},
)

ax.set_title("")

ax.set_xlabel(
    "Collection year",
    labelpad=12,
    fontweight="bold",
)

ax.set_ylabel(
    "Age group (years)",
    labelpad=12,
    fontweight="bold",
)

ax.tick_params(axis="both", labelsize=12)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")

for text in ax.texts:
    text.set_fontweight("bold")

cbar = ax.collections[0].colorbar

cbar.set_label(
    "Predicted positivity (%)",
    labelpad=12,
    fontweight="bold",
)

cbar.ax.invert_yaxis()

for label in cbar.ax.get_yticklabels():
    label.set_fontweight("bold")

plt.tight_layout()
fig.savefig(heatmap_png, dpi=1200, bbox_inches="tight")
fig.savefig(heatmap_pdf, dpi=1200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10.5, 5.8))

palette = sns.color_palette("viridis", n_colors=len(AGE_LABELS))

for age_group, color in zip(AGE_LABELS, palette):
    sub = prediction_grid[prediction_grid["age_group"] == age_group].copy()

    ax.plot(
        sub["collection_year"].astype(int),
        sub["predicted_positivity_percent"],
        marker="o",
        linewidth=1.8,
        markersize=4,
        color=color,
        label=age_group,
    )

ax.set_title("Age-associated alpha-gal IgE positivity gradient over time")
ax.set_xlabel("Collection year", fontweight="bold")
ax.set_ylabel("Predicted positivity (%)", fontweight="bold")
ax.set_xticks(YEAR_ORDER)
ax.set_xticklabels(YEAR_ORDER, rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")

ax.legend(
    title="Age group",
    frameon=False,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.tight_layout()
fig.savefig(lineplot_png, dpi=1200, bbox_inches="tight")
fig.savefig(lineplot_pdf, dpi=1200, bbox_inches="tight")
plt.show()
plt.close(fig)

ranking_rows = []

for year in YEAR_ORDER:
    sub = prediction_grid[
        prediction_grid["collection_year"].astype(int) == year
    ].copy()
    sub = sub.sort_values("predicted_positivity_percent")

    lowest_age_group = sub.iloc[0]["age_group"]
    highest_age_group = sub.iloc[-1]["age_group"]
    gradient_range = (
        sub["predicted_positivity_percent"].max()
        - sub["predicted_positivity_percent"].min()
    )

    ranking_rows.append(
        {
            "collection_year": year,
            "lowest_predicted_age_group": lowest_age_group,
            "highest_predicted_age_group": highest_age_group,
            "predicted_range_percentage_points": round(gradient_range, 1),
        }
    )

ranking = pd.DataFrame(ranking_rows)
ranking_csv = OUTPUT_DIR / "age_year_gradient_stability_summary.csv"
ranking.to_csv(ranking_csv, index=False)

most_common_highest = ranking["highest_predicted_age_group"].mode().iloc[0]
most_common_lowest = ranking["lowest_predicted_age_group"].mode().iloc[0]

range_median = ranking["predicted_range_percentage_points"].median()
range_min = ranking["predicted_range_percentage_points"].min()
range_max = ranking["predicted_range_percentage_points"].max()

print(f"Saved model comparison: {model_summary_csv}")
print(f"Saved predicted probabilities: {predicted_csv}")
print(f"Saved observed rates: {observed_csv}")
print(f"Saved heatmap PNG: {heatmap_png}")
print(f"Saved heatmap PDF: {heatmap_pdf}")
print(f"Saved line plot PNG: {lineplot_png}")
print(f"Saved line plot PDF: {lineplot_pdf}")
print(f"Saved gradient stability summary: {ranking_csv}")
print(f"Saved Results/Discussion text: {results_txt}")
print(f"Saved implausible DOB rows: {implausible_csv}")

print("\nModel comparison:")
print(model_output.to_string(index=False))

# Age-stratified (Table 3) and year-wise (Table 4) positivity and relative risks of alpha-gal


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, norm

input_file = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Data" / "aGal_IgE_20260406_non_repeated_patient_tests_2010_2025.csv"

OUTPUT_DIR = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

age_output_file = OUTPUT_DIR / "alpha_gal_nonrepeat_age_summary_2010_2025.csv"
year_output_file = OUTPUT_DIR / "alpha_gal_nonrepeat_year_summary_rowwise_2010_2025.csv"
implausible_output_file = OUTPUT_DIR / "alpha_gal_nonrepeat_implausible_dob_rows_2010_2025.csv"

BIRTH_YEAR_COL = "patient_dob_year"
DATE_COL = "collected_date"
IGE_COL = "result_for_analysis"

MIN_YEAR = 2010
MAX_YEAR = 2025
REF_YEAR = 2013

MIN_VALID_DOB_YEAR = 1900
MAX_VALID_DOB_YEAR = 2025
MIN_VALID_AGE = 0
MAX_VALID_AGE = 120

NEGATIVE_CUTOFF = 0.10
POSITIVE_CUTOFF = 0.35

def fmt_n_pct(n, denom):
    pct = 100 * n / denom if denom > 0 else np.nan
    return f"{int(n):,} ({pct:.1f})"


def fmt_mean_sd(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    if len(x) == 0:
        return ""
    return f"{x.mean():.1f} ({x.std(ddof=1):.1f})"


def rr_ci(a, b, c, d):
    """
    a = positive in comparison group
    b = non-positive in comparison group
    c = positive in reference group
    d = non-positive in reference group
    """
    if min(a, b, c, d) == 0:
        return "NA"

    risk1 = a / (a + b)
    risk0 = c / (c + d)
    rr = risk1 / risk0

    se = np.sqrt((1 / a) - (1 / (a + b)) + (1 / c) - (1 / (c + d)))
    lcl = np.exp(np.log(rr) - 1.96 * se)
    ucl = np.exp(np.log(rr) + 1.96 * se)

    return f"{rr:.2f} ({lcl:.2f}-{ucl:.2f})"


def two_prop_pvalue(a, b, c, d):
    """
    Two-proportion z test comparing positivity proportion
    in comparison group vs reference group.
    """
    n1 = a + b
    n0 = c + d

    if min(n1, n0) == 0:
        return "NA"

    p1 = a / n1
    p0 = c / n0
    p_pool = (a + c) / (n1 + n0)

    se = np.sqrt(p_pool * (1 - p_pool) * ((1 / n1) + (1 / n0)))

    if se == 0:
        return "NA"

    z = (p1 - p0) / se
    p = 2 * (1 - norm.cdf(abs(z)))

    return "<0.001" if p < 0.001 else f"{p:.3f}"


def format_pvalue(p):
    if pd.isna(p):
        return ""
    return "<0.001" if p < 0.001 else f"{p:.3f}"


df = pd.read_csv(input_file, low_memory=False)

tmp = df.copy()

tmp[BIRTH_YEAR_COL] = pd.to_numeric(tmp[BIRTH_YEAR_COL], errors="coerce")
tmp[IGE_COL] = pd.to_numeric(tmp[IGE_COL], errors="coerce")
tmp[DATE_COL] = pd.to_datetime(tmp[DATE_COL], errors="coerce")

tmp = tmp.dropna(subset=[BIRTH_YEAR_COL, DATE_COL, IGE_COL]).copy()

tmp["year"] = tmp[DATE_COL].dt.year
tmp = tmp[tmp["year"].between(MIN_YEAR, MAX_YEAR)].copy()

tmp["age"] = tmp["year"] - tmp[BIRTH_YEAR_COL]

tmp["implausible_dob"] = (
    tmp[BIRTH_YEAR_COL].isna()
    | ~tmp[BIRTH_YEAR_COL].between(MIN_VALID_DOB_YEAR, MAX_VALID_DOB_YEAR, inclusive="both")
    | (tmp[BIRTH_YEAR_COL] > tmp["year"])
    | (tmp["age"] < MIN_VALID_AGE)
    | (tmp["age"] > MAX_VALID_AGE)
)

implausible_rows = tmp[tmp["implausible_dob"]].copy()
implausible_rows.to_csv(implausible_output_file, index=False)

tmp = tmp[~tmp["implausible_dob"]].copy()

tmp["negative"] = tmp[IGE_COL] < NEGATIVE_CUTOFF
tmp["borderline"] = (tmp[IGE_COL] >= NEGATIVE_CUTOFF) & (tmp[IGE_COL] < POSITIVE_CUTOFF)
tmp["positive"] = tmp[IGE_COL] >= POSITIVE_CUTOFF
tmp["non_positive"] = ~tmp["positive"]

n_total = len(tmp)
n_pos = int(tmp["positive"].sum())
n_borderline = int(tmp["borderline"].sum())
n_neg = int(tmp["negative"].sum())

col_total = f"Total (N = {n_total:,})"
col_pos = f"Positive test result (n = {n_pos:,})"
col_borderline = f"Borderline test result (n = {n_borderline:,})"
col_neg = f"Negative test result (n = {n_neg:,})"

age_bins = [0, 10, 20, 30, 40, 50, 60, 70, np.inf]
age_labels = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]

tmp["age_group"] = pd.cut(
    tmp["age"],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True,
)

age_rows = []

positive_age = tmp.loc[tmp["positive"], "age"]
borderline_age = tmp.loc[tmp["borderline"], "age"]
negative_age = tmp.loc[tmp["negative"], "age"]
non_positive_age = tmp.loc[tmp["non_positive"], "age"]

if len(positive_age.dropna()) >= 2 and len(non_positive_age.dropna()) >= 2:
    p_age = ttest_ind(
        positive_age,
        non_positive_age,
        equal_var=False,
        nan_policy="omit",
    ).pvalue
else:
    p_age = np.nan

age_rows.append({
    "Characteristic": "Age, yrs, mean (SD)",
    col_total: fmt_mean_sd(tmp["age"]),
    col_pos: fmt_mean_sd(positive_age),
    col_borderline: fmt_mean_sd(borderline_age),
    col_neg: fmt_mean_sd(negative_age),
    "RR (95% CI)": "NA",
    "p-value": format_pvalue(p_age),
})

ref_age_group = "0-9"
ref = tmp[tmp["age_group"] == ref_age_group]
ref_pos = int(ref["positive"].sum())
ref_non_pos = int(ref["non_positive"].sum())

for grp in age_labels:
    sub = tmp[tmp["age_group"] == grp]

    total_n = len(sub)
    pos_n = int(sub["positive"].sum())
    borderline_n = int(sub["borderline"].sum())
    neg_n = int(sub["negative"].sum())
    non_pos_n = int(sub["non_positive"].sum())

    if grp == ref_age_group:
        rr_text = "1.00"
        p_text = ""
    else:
        rr_text = rr_ci(pos_n, non_pos_n, ref_pos, ref_non_pos)
        p_text = two_prop_pvalue(pos_n, non_pos_n, ref_pos, ref_non_pos)

    age_rows.append({
        "Characteristic": grp,
        col_total: f"{total_n:,}",
        col_pos: fmt_n_pct(pos_n, n_pos),
        col_borderline: fmt_n_pct(borderline_n, n_borderline),
        col_neg: fmt_n_pct(neg_n, n_neg),
        "RR (95% CI)": rr_text,
        "p-value": p_text,
    })

age_table = pd.DataFrame(age_rows)

year_rows = []

ref = tmp[tmp["year"] == REF_YEAR]
ref_pos = int(ref["positive"].sum())
ref_non_pos = int(ref["non_positive"].sum())

for yr in range(MIN_YEAR, MAX_YEAR + 1):
    sub = tmp[tmp["year"] == yr]

    total_n = len(sub)
    pos_n = int(sub["positive"].sum())
    borderline_n = int(sub["borderline"].sum())
    neg_n = int(sub["negative"].sum())
    non_pos_n = int(sub["non_positive"].sum())

    if yr < REF_YEAR:
        rr_text = "NA"
        p_text = "NA"
    elif yr == REF_YEAR:
        rr_text = "Ref"
        p_text = ""
    else:
        rr_text = rr_ci(pos_n, non_pos_n, ref_pos, ref_non_pos)
        p_text = two_prop_pvalue(pos_n, non_pos_n, ref_pos, ref_non_pos)

    year_rows.append({
        "Year": yr,
        "Total": f"{total_n:,}",
        "Positive test result, n (%)": fmt_n_pct(pos_n, total_n),
        "Borderline test result, n (%)": fmt_n_pct(borderline_n, total_n),
        "Negative test result, n (%)": fmt_n_pct(neg_n, total_n),
        "RR (95% CI)": rr_text,
        "p-value": p_text,
    })

year_table = pd.DataFrame(year_rows)

age_table.to_csv(age_output_file, index=False)
year_table.to_csv(year_output_file, index=False)

print(f"Rows after removing implausible DOB rows: {len(tmp):,}")
print(f"Positive results: {n_pos:,}")
print(f"Borderline results: {n_borderline:,}")
print(f"Negative results: {n_neg:,}")

print(f"Saved age table: {age_output_file}")
print(f"Saved year table: {year_output_file}")
print(f"Saved implausible DOB rows: {implausible_output_file}")

print("\nAge table:")
print(age_table.to_string(index=False))

print("\nYear table:")
print(year_table.to_string(index=False))

# Same-day categorical overlap of alpha-gal and meat sIgE (Table 5)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = BASE_DIR

INPUT_DIR = BASE_DIR / "03_outputs_v2" / "Single_Test_Spearman_PatientID_Date" / "Data"

OUTPUT_DIR = BASE_DIR / "03_outputs_v2" / "Categorical_Overlap_Copositivity"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
UNDERLYING_DIR = OUTPUT_DIR / "underlying_data"
REVIEW_DIR = OUTPUT_DIR / "manual_review"

for d in [TABLE_DIR, FIGURE_DIR, UNDERLYING_DIR, REVIEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PAIRED_FILES = {
    "Bos": INPUT_DIR / "Alpha-gal_vs_Bos_single_test_patient_id_collected_date_plot_data.csv",
    "Sus": INPUT_DIR / "Alpha-gal_vs_Sus_single_test_patient_id_collected_date_plot_data.csv",
    "Ovis": INPUT_DIR / "Alpha-gal_vs_Ovis_single_test_patient_id_collected_date_plot_data.csv",
}

PATIENT_COL = "patient_id"
DATE_COL = "collected_day"
ALPHA_COL = "Alpha-gal_ige"
POSITIVE_CUTOFF = 0.35

def fmt_pct(n, denom):
    return round(100 * n / denom, 1) if denom else np.nan


def classify_overlap(row):
    alpha_pos = row["alpha_gal_positive"]
    meat_pos = row["meat_positive"]

    if alpha_pos and meat_pos:
        return "Alpha-gal positive / Meat IgE positive"
    if alpha_pos and not meat_pos:
        return "Alpha-gal positive / Meat IgE non-positive"
    if not alpha_pos and meat_pos:
        return "Alpha-gal non-positive / Meat IgE positive"
    return "Both non-positive"


def make_stacked_bar(summary_df, assay, output_file):
    plot_order = [
        "Alpha-gal positive / Meat IgE positive",
        "Alpha-gal positive / Meat IgE non-positive",
        "Alpha-gal non-positive / Meat IgE positive",
        "Both non-positive",
    ]

    colors = {
        "Alpha-gal positive / Meat IgE positive": "#1b9e77",
        "Alpha-gal positive / Meat IgE non-positive": "#d95f02",
        "Alpha-gal non-positive / Meat IgE positive": "#7570b3",
        "Both non-positive": "#bdbdbd",
    }

    data = summary_df.set_index("Category").loc[plot_order].copy()

    fig, ax = plt.subplots(figsize=(10, 2.8))

    left = 0
    total = int(data["Count"].sum())

    for category, row in data.iterrows():
        count = int(row["Count"])
        pct = row["Percent"]

        ax.barh(
            y=[0],
            width=[pct],
            left=[left],
            color=colors[category],
            edgecolor="white",
            linewidth=1.2,
            label=category,
        )

        if pct >= 4:
            ax.text(
                left + pct / 2,
                0,
                f"{count:,}\n{pct:.1f}%",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if category != "Both non-positive" else "black",
                fontweight="bold",
            )
        else:
            ax.text(
                left + pct + 0.5,
                0,
                f"{count:,} ({pct:.1f}%)",
                ha="left",
                va="center",
                fontsize=8,
                color="black",
            )

        left += pct

    ax.set_xlim(0, 100)
    ax.set_yticks([])
    ax.set_xlabel("Percent of same-day paired single-test observations")
    ax.set_title(f"Categorical overlap of Alpha-gal IgE and {assay} IgE\nN = {total:,}")

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.35),
        ncol=2,
        frameon=False,
        fontsize=8,
    )

    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25)

    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    fig.savefig(output_file.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


combined_rows = []
interpretation_rows = []

for assay, input_file in PAIRED_FILES.items():
    meat_col = f"{assay}_ige"

    df = pd.read_csv(input_file, low_memory=False)

    df[PATIENT_COL] = df[PATIENT_COL].astype("string").str.strip()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce").dt.date
    df[ALPHA_COL] = pd.to_numeric(df[ALPHA_COL], errors="coerce")
    df[meat_col] = pd.to_numeric(df[meat_col], errors="coerce")

    df = df.dropna(subset=[PATIENT_COL, DATE_COL, ALPHA_COL, meat_col]).copy()
    df = df[df[PATIENT_COL] != ""].copy()

    duplicate_mask = df.duplicated(subset=[PATIENT_COL, DATE_COL], keep=False)
    ambiguous_df = df[duplicate_mask].copy()

    ambiguous_file = REVIEW_DIR / f"Alpha-gal_vs_{assay}_ambiguous_duplicate_patient_date_pairs.csv"
    ambiguous_df.to_csv(ambiguous_file, index=False)

    analytic = df[~duplicate_mask].copy()
    analytic = analytic.drop_duplicates(subset=[PATIENT_COL, DATE_COL], keep="first").copy()

    analytic["alpha_gal_positive"] = analytic[ALPHA_COL] >= POSITIVE_CUTOFF
    analytic["meat_positive"] = analytic[meat_col] >= POSITIVE_CUTOFF

    analytic["overlap_category"] = analytic.apply(classify_overlap, axis=1)

    total_n = len(analytic)

    category_order = [
        "Alpha-gal positive / Meat IgE positive",
        "Alpha-gal positive / Meat IgE non-positive",
        "Alpha-gal non-positive / Meat IgE positive",
        "Both non-positive",
    ]

    summary_rows = []

    for category in category_order:
        category_df = analytic[analytic["overlap_category"] == category].copy()
        count = len(category_df)
        percent = fmt_pct(count, total_n)

        category_safe = (
            category.replace("Alpha-gal", "alpha_gal")
            .replace("Meat IgE", assay.lower())
            .replace(" ", "_")
            .replace("/", "and")
            .replace("-", "_")
        )

        category_file = UNDERLYING_DIR / f"Alpha-gal_vs_{assay}_{category_safe}.csv"
        category_df.to_csv(category_file, index=False)

        summary_rows.append({
            "Comparison": f"Alpha-gal vs {assay}",
            "Category": category,
            "Count": count,
            "Percent": percent,
            "Total paired observations": total_n,
            "Underlying data file": str(category_file),
        })

    summary_df = pd.DataFrame(summary_rows)

    alpha_pos_n = int(analytic["alpha_gal_positive"].sum())
    meat_pos_n = int(analytic["meat_positive"].sum())
    both_pos_n = int((analytic["alpha_gal_positive"] & analytic["meat_positive"]).sum())

    pct_alpha_pos_also_meat_pos = fmt_pct(both_pos_n, alpha_pos_n)
    pct_meat_pos_also_alpha_pos = fmt_pct(both_pos_n, meat_pos_n)

    summary_df["Alpha-gal positive total"] = alpha_pos_n
    summary_df[f"{assay} positive total"] = meat_pos_n
    summary_df["Both positive total"] = both_pos_n
    summary_df[f"% alpha-gal positive also {assay} positive"] = pct_alpha_pos_also_meat_pos
    summary_df[f"% {assay} positive also alpha-gal positive"] = pct_meat_pos_also_alpha_pos
    summary_df["Ambiguous duplicate patient/date rows flagged"] = len(ambiguous_df)
    summary_df["Ambiguous duplicate file"] = str(ambiguous_file)

    summary_file = TABLE_DIR / f"Alpha-gal_vs_{assay}_categorical_overlap_summary.csv"
    summary_df.to_csv(summary_file, index=False)

    combined_rows.extend(summary_rows)

    combined_rows.append({
        "Comparison": f"Alpha-gal vs {assay}",
        "Category": "% alpha-gal positive also meat positive",
        "Count": both_pos_n,
        "Percent": pct_alpha_pos_also_meat_pos,
        "Total paired observations": alpha_pos_n,
        "Underlying data file": "",
    })

    combined_rows.append({
        "Comparison": f"Alpha-gal vs {assay}",
        "Category": "% meat positive also alpha-gal positive",
        "Count": both_pos_n,
        "Percent": pct_meat_pos_also_alpha_pos,
        "Total paired observations": meat_pos_n,
        "Underlying data file": "",
    })

    figure_file = FIGURE_DIR / f"Alpha-gal_vs_{assay}_categorical_overlap_stacked_bar.png"
    make_stacked_bar(summary_df, assay, figure_file)

    interpretation_rows.append({
        "Comparison": f"Alpha-gal vs {assay}",
        "Total paired observations": total_n,
        "Both positive": both_pos_n,
        "Both positive percent": fmt_pct(both_pos_n, total_n),
        "Alpha-gal positive total": alpha_pos_n,
        f"% alpha-gal positive also {assay} positive": pct_alpha_pos_also_meat_pos,
        f"{assay} positive total": meat_pos_n,
        f"% {assay} positive also alpha-gal positive": pct_meat_pos_also_alpha_pos,
        "Ambiguous duplicate patient/date rows flagged": len(ambiguous_df),
    })

combined_summary = pd.DataFrame(combined_rows)
combined_summary_file = TABLE_DIR / "combined_alpha_gal_vs_meat_categorical_overlap_summary.csv"
combined_summary.to_csv(combined_summary_file, index=False)

interpretation_df = pd.DataFrame(interpretation_rows)
interpretation_table_file = TABLE_DIR / "combined_alpha_gal_vs_meat_copositivity_metrics.csv"
interpretation_df.to_csv(interpretation_table_file, index=False)

print(f"Saved combined summary table: {combined_summary_file}")
print(f"Saved combined copositivity metrics: {interpretation_table_file}")
print(f"Saved per-comparison tables to: {TABLE_DIR}")
print(f"Saved figures to: {FIGURE_DIR}")
print(f"Saved category-level underlying data to: {UNDERLYING_DIR}")
print(f"Saved ambiguous duplicate patient/date rows to: {REVIEW_DIR}")

print("\nCopositivity metrics:")
print(interpretation_df.to_string(index=False))

# Adjusted odds of alpha-gal (Supplementary figure 2) and multivariable logistic regression (Supplementary table 2)


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

input_file = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Data" / "aGal_IgE_20260406_non_repeated_patient_tests_2010_2025.csv"

OUTPUT_DIR = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

regression_output_file = OUTPUT_DIR / "multivariable_logistic_regression_alpha_gal.csv"
age_forest_data_file = OUTPUT_DIR / "adjusted_age_odds_ratio_forest_plot_data.csv"
forest_png_file = OUTPUT_DIR / "adjusted_age_odds_ratio_forest_plot_linear.png"
forest_pdf_file = OUTPUT_DIR / "adjusted_age_odds_ratio_forest_plot_linear.pdf"

BIRTH_YEAR_COL = "patient_dob_year"
DATE_COL = "collected_date"
IGE_COL = "result_for_analysis"

MIN_YEAR = 2010
MAX_YEAR = 2025
MODEL_MIN_YEAR = 2013
REF_YEAR = 2013

MIN_VALID_DOB_YEAR = 1900
MAX_VALID_DOB_YEAR = 2025
MIN_VALID_AGE = 0
MAX_VALID_AGE = 120

POSITIVE_CUTOFF = 0.35

AGE_LABELS = ["0–9", "10–19", "20–29", "30–39", "40–49", "50–59", "60–69", "70+"]
AGE_BINS = [0, 10, 20, 30, 40, 50, 60, 70, np.inf]
REF_AGE_GROUP = "0–9"

MODEL_FORMULA = (
    'alpha_gal_positive ~ '
    'C(age_group, Treatment(reference="0–9")) + '
    'C(collection_year, Treatment(reference=2013))'
)

def extract_category(term):
    match = re.search(r"\[T\.(.*)\]", term)
    return match.group(1) if match else ""


df = pd.read_csv(input_file, low_memory=False)

tmp = df.copy()

tmp[BIRTH_YEAR_COL] = pd.to_numeric(tmp[BIRTH_YEAR_COL], errors="coerce")
tmp[IGE_COL] = pd.to_numeric(tmp[IGE_COL], errors="coerce")
tmp[DATE_COL] = pd.to_datetime(tmp[DATE_COL], errors="coerce")

tmp = tmp.dropna(subset=[BIRTH_YEAR_COL, DATE_COL, IGE_COL]).copy()

tmp["collection_year"] = tmp[DATE_COL].dt.year
tmp = tmp[tmp["collection_year"].between(MIN_YEAR, MAX_YEAR)].copy()

tmp["age"] = tmp["collection_year"] - tmp[BIRTH_YEAR_COL]

tmp["implausible_dob"] = (
    ~tmp[BIRTH_YEAR_COL].between(MIN_VALID_DOB_YEAR, MAX_VALID_DOB_YEAR, inclusive="both")
    | (tmp[BIRTH_YEAR_COL] > tmp["collection_year"])
    | (tmp["age"] < MIN_VALID_AGE)
    | (tmp["age"] > MAX_VALID_AGE)
)

tmp = tmp[~tmp["implausible_dob"]].copy()

tmp["alpha_gal_positive"] = (tmp[IGE_COL] >= POSITIVE_CUTOFF).astype(int)

tmp["age_group"] = pd.cut(
    tmp["age"],
    bins=AGE_BINS,
    labels=AGE_LABELS,
    right=False,
    include_lowest=True,
)

model_df = tmp[tmp["collection_year"] >= MODEL_MIN_YEAR].copy()
model_df = model_df.dropna(subset=["alpha_gal_positive", "age_group", "collection_year"]).copy()

model_df["age_group"] = pd.Categorical(
    model_df["age_group"],
    categories=AGE_LABELS,
    ordered=True,
)

model_df["collection_year"] = model_df["collection_year"].astype(int)

model = smf.logit(MODEL_FORMULA, data=model_df).fit()

print(model.summary())

n_patients = len(model_df)
n_positive = int(model_df["alpha_gal_positive"].sum())
n_non_positive = n_patients - n_positive

print(f"\nNumber of patients included: {n_patients:,}")
print(f"Number positive: {n_positive:,}")
print(f"Number non-positive: {n_non_positive:,}")
print(f"Model formula: {MODEL_FORMULA}")
print(f"Reference age group: {REF_AGE_GROUP}")
print(f"Reference collection year: {REF_YEAR}")
print("Years excluded from adjusted model: 2010–2012")

params = model.params
conf = model.conf_int()
pvals = model.pvalues

rows = []

for term in params.index:
    if term == "Intercept":
        continue

    adjusted_or = np.exp(params.loc[term])
    ci_lower = np.exp(conf.loc[term, 0])
    ci_upper = np.exp(conf.loc[term, 1])
    p_value = pvals.loc[term]

    if term.startswith("C(age_group"):
        predictor = "age_group"
        category = extract_category(term)
        reference_category = REF_AGE_GROUP
    elif term.startswith("C(collection_year"):
        predictor = "collection_year"
        category = extract_category(term)
        reference_category = str(REF_YEAR)
    else:
        predictor = term
        category = ""
        reference_category = ""

    rows.append({
        "predictor": predictor,
        "category": category,
        "reference_category": reference_category,
        "adjusted_odds_ratio": adjusted_or,
        "ci_lower_95": ci_lower,
        "ci_upper_95": ci_upper,
        "p_value": p_value,
    })

regression_table = pd.DataFrame(rows)
regression_table.to_csv(regression_output_file, index=False)

print(f"\nSaved regression table: {regression_output_file}")

age_plot_df = regression_table[regression_table["predictor"] == "age_group"].copy()

age_plot_df["category"] = pd.Categorical(
    age_plot_df["category"],
    categories=AGE_LABELS[1:],
    ordered=True,
)

age_plot_df = age_plot_df.sort_values("category", ascending=False)

age_plot_export = age_plot_df[
    [
        "category",
        "reference_category",
        "adjusted_odds_ratio",
        "ci_lower_95",
        "ci_upper_95",
        "p_value",
    ]
].copy()

age_plot_export.to_csv(age_forest_data_file, index=False)

print(f"Saved age forest plot data: {age_forest_data_file}")

y_pos = np.arange(len(age_plot_df))

or_values = age_plot_df["adjusted_odds_ratio"].to_numpy()
ci_lower = age_plot_df["ci_lower_95"].to_numpy()
ci_upper = age_plot_df["ci_upper_95"].to_numpy()

xerr = np.vstack([
    or_values - ci_lower,
    ci_upper - or_values,
])

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 2,
})

fig, ax = plt.subplots(figsize=(7.2, 4.8))

ax.errorbar(
    or_values,
    y_pos,
    xerr=xerr,
    fmt="o",
    color="black",
    ecolor="black",
    elinewidth=1.2,
    capsize=3,
    markersize=5,
)

ax.axvline(1.0, color="red", linestyle="--", linewidth=2)

ax.set_xlim(0.8, 4.2)
ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(["1", "2", "3", "4"])

ax.set_yticks(y_pos)
ax.set_yticklabels(age_plot_df["category"])
ax.set_xlabel("Adjusted odds ratio (95% CI)", fontweight="bold", fontsize=15, labelpad=10)
ax.set_ylabel("Age group (years)", fontweight="bold", fontsize=15, labelpad=10)
ax.set_title("", fontweight="bold")

ax.grid(False)

fig.tight_layout()

fig.savefig(forest_png_file, dpi=1200, bbox_inches="tight")
fig.savefig(forest_pdf_file, dpi=1200, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved forest plot PNG: {forest_png_file}")
print(f"Saved forest plot PDF: {forest_pdf_file}")

print("\nClean regression table:")
print(regression_table.to_string(index=False))

print("\nAge forest plot data:")
print(age_plot_export.to_string(index=False))

In [ ]:
RESTRICTED_MIN_YEAR = 2015
RESTRICTED_MAX_YEAR = 2020
RESTRICTED_REF_YEAR = 2015

restricted_output_csv = OUTPUT_DIR / "Supplementary_Table_7_restricted_logistic_2015_2020.csv"
restricted_output_xlsx = OUTPUT_DIR / "Supplementary_Table_7_restricted_logistic_2015_2020.xlsx"

RESTRICTED_MODEL_FORMULA = (
    'alpha_gal_positive ~ '
    'C(age_group, Treatment(reference="0–9")) + '
    'C(collection_year, Treatment(reference=2015))'
)

restricted_model_df = tmp[
    tmp["collection_year"].between(RESTRICTED_MIN_YEAR, RESTRICTED_MAX_YEAR)
].copy()

restricted_model_df = restricted_model_df.dropna(
    subset=["alpha_gal_positive", "age_group", "collection_year"]
).copy()

restricted_model_df["age_group"] = pd.Categorical(
    restricted_model_df["age_group"],
    categories=AGE_LABELS,
    ordered=True,
)

restricted_model_df["collection_year"] = restricted_model_df["collection_year"].astype(int)

restricted_model = smf.logit(
    RESTRICTED_MODEL_FORMULA,
    data=restricted_model_df
).fit()

print(restricted_model.summary())

n_restricted_patients = len(restricted_model_df)
n_restricted_positive = int(restricted_model_df["alpha_gal_positive"].sum())
n_restricted_non_positive = n_restricted_patients - n_restricted_positive

print("\n2015–2020 restricted-period logistic regression sensitivity analysis")
print(f"Number of patients included: {n_restricted_patients:,}")
print(f"Number positive: {n_restricted_positive:,}")
print(f"Number non-positive: {n_restricted_non_positive:,}")
print(f"Model formula: {RESTRICTED_MODEL_FORMULA}")
print(f"Reference age group: {REF_AGE_GROUP}")
print(f"Reference collection year: {RESTRICTED_REF_YEAR}")

restricted_params = restricted_model.params
restricted_conf = restricted_model.conf_int()
restricted_pvals = restricted_model.pvalues

restricted_rows = []

for term in restricted_params.index:
    if term == "Intercept":
        continue

    adjusted_or = np.exp(restricted_params.loc[term])
    ci_lower = np.exp(restricted_conf.loc[term, 0])
    ci_upper = np.exp(restricted_conf.loc[term, 1])
    p_value = restricted_pvals.loc[term]

    if term.startswith("C(age_group"):
        predictor = "age_group"
        category = extract_category(term)
        reference_category = REF_AGE_GROUP
    elif term.startswith("C(collection_year"):
        predictor = "collection_year"
        category = extract_category(term)
        reference_category = str(RESTRICTED_REF_YEAR)
    else:
        predictor = term
        category = ""
        reference_category = ""

    restricted_rows.append({
        "predictor": predictor,
        "category": category,
        "reference_category": reference_category,
        "adjusted_odds_ratio": adjusted_or,
        "ci_lower_95": ci_lower,
        "ci_upper_95": ci_upper,
        "p_value": p_value,
    })

restricted_regression_table = pd.DataFrame(restricted_rows)

restricted_regression_table.to_csv(restricted_output_csv, index=False)

try:
    restricted_regression_table.to_excel(restricted_output_xlsx, index=False)
    print(f"\nSaved restricted regression table Excel: {restricted_output_xlsx}")
except ImportError:
    print("\nExcel export skipped: install openpyxl or xlsxwriter to enable .xlsx export.")

print(f"Saved restricted regression table CSV: {restricted_output_csv}")

main_age_effects = regression_table[
    regression_table["predictor"] == "age_group"
][["category", "adjusted_odds_ratio"]].copy()

restricted_age_effects = restricted_regression_table[
    restricted_regression_table["predictor"] == "age_group"
][["category", "adjusted_odds_ratio"]].copy()

main_age_effects["main_direction"] = np.where(
    main_age_effects["adjusted_odds_ratio"] > 1,
    "higher odds than 0–9",
    np.where(main_age_effects["adjusted_odds_ratio"] < 1, "lower odds than 0–9", "same odds as 0–9")
)

restricted_age_effects["restricted_direction"] = np.where(
    restricted_age_effects["adjusted_odds_ratio"] > 1,
    "higher odds than 0–9",
    np.where(restricted_age_effects["adjusted_odds_ratio"] < 1, "lower odds than 0–9", "same odds as 0–9")
)

age_direction_check = main_age_effects.merge(
    restricted_age_effects,
    on="category",
    suffixes=("_main_2013_2025", "_restricted_2015_2020")
)

age_direction_check["same_direction"] = (
    age_direction_check["main_direction"] == age_direction_check["restricted_direction"]
)

print("\nDirection check for age-group effects vs main 2013–2025 model:")
print(age_direction_check.to_string(index=False))

print("\nRestricted-period regression table:")
print(restricted_regression_table.to_string(index=False))

# Multiple-comparison adjustment FDR (Supplementary table 6)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

BASE_DIR = Path.cwd().resolve()

while not {"01_raw_data", "02_clean_data", "03_outputs_v2", "notebooks"}.issubset(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()}
):
    BASE_DIR = BASE_DIR.parent

OUTPUT_DIR = BASE_DIR / "03_outputs_v2" / "Alpha-gal" / "Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

age_rr_file = OUTPUT_DIR / "alpha_gal_nonrepeat_age_summary_2010_2025.csv"
year_rr_file = OUTPUT_DIR / "alpha_gal_nonrepeat_year_summary_rowwise_2010_2025.csv"
logistic_file = OUTPUT_DIR / "multivariable_logistic_regression_alpha_gal.csv"

fdr_output_file = OUTPUT_DIR / "multiple_comparison_fdr_adjustment_summary.csv"


spearman_pvalues = {
    "alpha-gal vs beef IgE": 0.001,
    "alpha-gal vs pork IgE": 0.001,
    "alpha-gal vs lamb/mutton IgE": 0.001,
}

secondary_pvalues = {
    "harmonic regression overall seasonality LRT, unadjusted model 2010-2025": 0.001,
    "harmonic regression overall seasonality LRT, year-adjusted model 2013-2025": 0.001,
    "Kaplan-Meier log-rank test by initial IgE intensity": 0.001,
}

INCLUDE_LOGISTIC_REGRESSION = True

def parse_pvalue(x):
    """
    Convert P-values like '<0.001', '0.023', '', 'NA' to numeric values.
    If exact P-values are available, use exact values.
    Here '<0.001' is conservatively treated as 0.001.
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "" or x.upper() in {"NA", "NAN", "NONE"}:
        return np.nan

    if x.startswith("<"):
        return float(x.replace("<", "").strip())

    return float(x)


def format_pvalue(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def add_test(rows, analysis_family, comparison, raw_p_value):
    p = parse_pvalue(raw_p_value)

    if pd.notna(p):
        rows.append({
            "analysis_family": analysis_family,
            "comparison": comparison,
            "raw_p_value": p,
        })


def find_pvalue_column(df):
    possible_cols = ["p-value", "p_value", "p", "P-value", "P_value"]

    for col in possible_cols:
        if col in df.columns:
            return col

    raise ValueError(f"No P-value column found. Columns are: {list(df.columns)}")


rows = []

age_table = pd.read_csv(age_rr_file)
age_p_col = find_pvalue_column(age_table)

age_groups = ["10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]

for age_group in age_groups:
    sub = age_table[
        age_table["Characteristic"].astype(str).str.strip() == age_group
    ]

    if len(sub) == 1:
        add_test(
            rows,
            analysis_family="Age-stratified relative risk comparisons",
            comparison=f"{age_group} vs 0-9",
            raw_p_value=sub[age_p_col].iloc[0],
        )

year_table = pd.read_csv(year_rr_file)
year_p_col = find_pvalue_column(year_table)

year_table["Year"] = pd.to_numeric(year_table["Year"], errors="coerce")

for year in range(2014, 2026):
    sub = year_table[year_table["Year"] == year]

    if len(sub) == 1:
        add_test(
            rows,
            analysis_family="Year-wise relative risk comparisons",
            comparison=f"{year} vs 2013",
            raw_p_value=sub[year_p_col].iloc[0],
        )

for comparison, p_value in spearman_pvalues.items():
    add_test(
        rows,
        analysis_family="Pairwise Spearman correlations",
        comparison=comparison,
        raw_p_value=p_value,
    )

if INCLUDE_LOGISTIC_REGRESSION and logistic_file.exists():
    logistic_table = pd.read_csv(logistic_file)

    if "p_value" not in logistic_table.columns:
        raise ValueError(
            f"Expected 'p_value' in logistic regression file. "
            f"Columns are: {list(logistic_table.columns)}"
        )

    for _, r in logistic_table.iterrows():
        predictor = r.get("predictor", "")
        category = r.get("category", "")
        reference = r.get("reference_category", "")

        add_test(
            rows,
            analysis_family="Multivariable logistic regression",
            comparison=f"{predictor}: {category} vs {reference}",
            raw_p_value=r["p_value"],
        )

for comparison, p_value in secondary_pvalues.items():
    add_test(
        rows,
        analysis_family="Optional secondary analyses",
        comparison=comparison,
        raw_p_value=p_value,
    )

fdr_table = pd.DataFrame(rows)

if fdr_table.empty:
    raise ValueError(
        "No valid P-values were found. Check input files and manually entered P-values."
    )

reject, adjusted_pvalues, _, _ = multipletests(
    fdr_table["raw_p_value"].to_numpy(),
    alpha=0.05,
    method="fdr_bh",
)

fdr_table["adjusted_p_value_BH_FDR"] = adjusted_pvalues
fdr_table["significant_after_FDR"] = reject

fdr_table["raw_p_value_formatted"] = fdr_table["raw_p_value"].apply(format_pvalue)
fdr_table["adjusted_p_value_BH_FDR_formatted"] = fdr_table[
    "adjusted_p_value_BH_FDR"
].apply(format_pvalue)

fdr_table = fdr_table[
    [
        "analysis_family",
        "comparison",
        "raw_p_value",
        "raw_p_value_formatted",
        "adjusted_p_value_BH_FDR",
        "adjusted_p_value_BH_FDR_formatted",
        "significant_after_FDR",
    ]
].copy()

fdr_table.to_csv(fdr_output_file, index=False)


print(f"Saved FDR summary table: {fdr_output_file}")
print(f"Number of P-values included: {n_tests:,}")
print(f"Nominally significant before FDR: {n_raw_sig:,}")
print(f"Significant after BH FDR correction: {n_fdr_sig:,}")



print("\nFDR-adjusted results:")
print(fdr_table.to_string(index=False))
